In [ ]:
%%html
<style>
toc {
    position: fixed;
    top: 50px;
    left: 20px;
    width: 250px;
    max-height: 80vh;
    overflow-y: auto;
    background: #f8f8f8;
    padding: 15px;
    border: 1px solid #ddd;
    border-radius: 5px;
    z-index: 1000;
}

</style>

<script>
// Auto-generate TOC from headers
document.addEventListener("DOMContentLoaded", function() {
    const headers = document.querySelectorAll('h1, h2, h3');
    const toc = document.getElementById('toc');
    
    if (toc && headers.length > 0) {
        let tocHTML = '<h3>Contents</h3><ul>';
        headers.forEach((header, index) => {
            const id = 'section-' + index;
            header.id = id;
            const level = parseInt(header.tagName[1]);
            const indent = (level - 1) * 20;
            tocHTML += `<li style="margin-left: ${indent}px">
                <a href="#${id}">${header.textContent}</a>
            </li>`;
        });
        tocHTML += '</ul>';
        toc.innerHTML = tocHTML;
    }
});
</script>

In [ ]:
Q4_genes = [
    'FABP5', 'PDHA1', 'TRIP13', 'AIM2', 'SELI', 'SLC19A1', 'LARS2', 'OPN3',
    'ASPM', 'CCT2', 'UBE2I', 'STK6', 'FLJ13052', 'LAS1L', 'BIRC5', 'RFC4',
    'CKS1B', 'CKAP1', 'MGC57827', 'DKFZp779O175', 'PFN1', 'ILF3', 'IFI16',
    'TBRG4', 'PAPD1', 'EIF2C2', 'MGC4308', 'ENO1', 'DSG2', 'C6orf173',
    'EXOSC4', 'TAGLN2', 'RUVBL1', 'ALDOA', 'CPSF3', 'NA', 'MGC15606',
    'LGALS1', 'RAD18', 'SNX5', 'PSMD4', 'RAN', 'KIF14', 'CBX3', 'TMPO',
    'DKFZp586L0724', 'WEE1', 'ROBO1', 'TCOF1', 'YWHAZ', 'MPHOSPH1'
]

# Q1 genes (low expression associated with poor prognosis)
Q1_genes = [
    'GNG10', 'NA', 'PNPLA4', 'NA', 'KIAA1754', 'AHCYL1', 'MCLC', 'EVI5',
    'AD-020', 'NA', 'PARG1', 'CTBS', 'UBE2R2', 'FUCA1', 'RFP2', 'FLJ20489',
    'NA', 'LTBP1', 'TRIM33'
]

# Combined list (all genes, excluding NA entries)
Q4_genes_filtered = [g for g in Q4_genes if g != 'NA']
Q1_genes_filtered = [g for g in Q1_genes if g != 'NA']
GEP70_genes = Q4_genes_filtered + Q1_genes_filtered


CUSTOM_PATHWAYS = {
    "NK1_sig": ["MYOM2", "SPON2", "CLIC3", "LAIR2", "AKR1C3", "IGFBP7", "FGFBP2", "FCER1G", "GZMB", "CHST2", "PRF1", "CD38", "DAB2", "NKG7", "CX3CR1", "KLRB1", "CD247", "PLAC8", "TMIGD2", "ADGRG1"],
    "NK2_sig": ["GZMK", "XCL2", "CMC1", "SELL", "XCL1", "KLRC1", "TCF7", "CD44", "GPR183", "TPT1", "RPS2", "RPLP1", "EEF1A1", "RPS18", "RPS15A", "RPS23", "RPS12", "RPS8", "RPL13", "RPL13A"],
    "NK3_sig": ["KLRC2", "CD3E", "GZMH", "CCL5", "IL32", "S100A6", "VIM", "LINC01871", "PTMS", "S100A4", "LAG3", "ITGB1", "TMSB4X", "RAP2A", "CD2", "CD52", "MYO6", "LGALS1", "HLA-B", "PRDM1"],
    "Cytototxicity": ["FASLG", "GSDMD", "GZMB", "NKG7", "PRF1", "GZMA", "GZMH", "TNFSF10"],
    "Cytokine_Receptors": ["TGFBR1", "IL12RB1", "IL2RG", "TGFBR3", "TGFBR2", "IL10RA", "IL18R1", "IL18RAP", "IL2RB", "IL10RB"],
    "Cyto_Chemokines": ["IFNG", "CCL4", "CCL4L2", "CCL5", "IL32", "IL16", "CCL3", "XCL2", "XCL1", "FLT3LG"],
    "Activating_Receptors": ["KLRK1", "KLRC2", "NCR1", "CD160", "NCR3", "SLAMF6", "SLAMF7"],
    "Inhibitory_Receptors": ["KLRC1", "CD300A", "TIGIT", "KIR3DL2", "KIR3DL1", "KIR2DL3", "KIR2DL1", "KLRB1", "SIGLEC7", "HAVCR2"],
    "IA_Interferon_Sig": ["IFI44L", "ISG15", "XAF1", "IFI6", "MX1", "LY6E", "EIF2AK2", "STAT1", "MX2", "IFIT3", "ISG20", "OAS1", "IFI44", "IRF7", "IFIT1", "OAS3", "IFI35", "OAS2", "IFI27"],
    "IA_Tcell_Cytotox_Sig": ["GZMA", "GZMB", "GZMH", "GZMK", "GZMH", "GNLY", "PRF1", "IFNG", "TNF", "SERPINB1", "SERPINB6", "SERPINB9", "CTSA", "CTSB", "CTSC", "CTSD", "CTSW", "CST3", "CST7", "CSTB", "LAMP1", "LAMP3", "CAPN2", "GZMM", "CTSH", "CAPN2", "PLEK"],
    "IA_Tcell_Naive_Sig": ["IL7R", "CCR7", "SELL", "FOXO1", "KLF2", "KLF3", "LEF1", "TCF7", "ACTN1", "FOXP1", "BTG1", "BTG2", "TOB1"],
    "IA_Tcell_Exh_Sig": ["PDCD1", "LAYN", "HAVCR2", "LAG3", "CD244", "CTLA4", "LILRB1", "TIGIT", "TOX", "VSIR", "BTLA", "ENTPD1", "CD160", "LAIR1"],
    "IA_Tcell_SASP_Sig": ["ACVR1B", "ANG", "ANGPT1", "ANGPTL4", "AREG", "AXL", "BEX3", "BMP2", "BMP6", "C3", "CCL1", "CCL13", "CCL16", "CCL2", "CCL20", "CCL24", "CCL26", "CCL3", "CCL3L1", "CCL4", "CCL5", "CCL7", "CCL8", "CD55", "CD9", "CSF1", "CSF2", "CSF2RB", "CST4", "CTNNB1", "CTSB", "CXCL1", "CXCL10", "CXCL12", "CXCL16", "CXCL2", "CXCL3", "CXCL8", "CXCR2", "DKK1", "EDN1", "EGF", "EGFR", "EREG", "ESM1", "ETS2", "FAS", "FGF1", "FGF2", "FGF7", "GDF15", "GEM", "GMFG", "HGF", "HMGB1", "ICAM1", "ICAM3", "IGF1", "IGFBP1", "IGFBP2", "IGFBP3", "IGFBP4", "IGFBP5", "IGFBP6", "IGFBP7", "IL10", "IL13", "IL15", "IL18", "IL1A", "IL1B", "IL2", "IL32", "IL6", "IL6ST", "IL7", "INHA", "IQGAP2", "ITGA2", "ITPKA", "JUN", "KITLG", "LCP1", "MIF", "MMP1", "MMP10", "MMP12", "MMP13", "MMP14", "MMP2", "MMP3", "MMP9", "NAP1L4", "NRG1", "PAPPA", "PECAM1", "PGF", "PIGF", "PLAT", "PLAU", "PLAUR", "PTBP1", "PTGER2", "PTGES", "RPS6KA5", "SCAMP4", "SELPLG", "SEMA3F", "SERPINB4", "SERPINE1", "SERPINE2", "SPP1", "SPX", "TIMP2", "TNF", "TNFRSF10C", "TNFRSF11B", "TNFRSF1A", "TNFRSF1B", "TUBGCP2", "VEGFA", "VEGFC", "VGF", "WNT16", "WNT2"],
    "SwiftSeq_CTC_Sig": ['CRIP1', 'HLA-DRA', 'PTPRC', 'MS4A1', 'AC233755.2', 'TAGLN2', 'AHNAK', 'MIR155HG', 'MYADM', 'CD52', 'S100A10', 'TUBA1A', 'CCL3', 'FLNA', 'CD79A', 'CD44', 'TMSB4X', 'CD81', 'FCMR', 'SOCS3', 'PCSK1N', 'HIST1H2BF', 'ADAM8', 'RHOC', 'S100A4', 'ANXA2', 'CLEC2B', 'ZYX', 'PTMS', 'AC007952.4', 'EMP3', 'S100A6', 'ADGRE5', 'STX2', 'IL27RA', 'LMNA', 'COTL1', 'LGALS1', 'SGK1', 'EZR', 'MTSS1', 'LINC01480', 'STAT6', 'CD99', 'MT2A', 'TRIM22', 'KLF4', 'KLF9', 'BIRC3', 'DOCK11', 'AL627171.2', 'LCP1', 'RAB29', 'CBX6', 'PLEKHO1', 'AC103591.3', 'HSPA1B', 'Z99127.4', 'STBD1', 'IL15', 'PTPRCAP', 'PEA15', 'DAAM1', 'MAL', 'JCHAIN', 'TMEM43', 'S100A11', 'IFI27', 'BHLHE40', 'IZUMO4', 'MIR22HG', 'YWHAH', 'AL133415.1', 'LAPTM5', 'SQLE', 'KLF6', 'DDIT4', 'JAZF1', 'KCNN4', 'CTSH', 'APOBEC3G', 'FAM89B', 'TYMP', 'PLEC', 'WNT10A', 'HIVEP1', 'TEX9', 'RAB31', 'AC009133.3', 'RND3', 'LAT2', 'PLAAT2', 'ANTXR2', 'LRRC8C', 'BBC3', 'DNAAF1', 'AL138963.4', 'TMC8', 'LTK', 'CAPG'],
    "UAMS70_UP": Q4_genes,
    "UAMS70_Down": Q1_genes
}

CLUSTERS_ANALYSIS=None
FILE_NAME="Plasma_Cells"
ADDITIONAL_SUBDIR="BULK_test/"
ANALYSIS_DESC="Differential Expression across Bulk plasma cells cells, with respect to high vs low AFR Ancestry."
COMPARTMENT_DESCRIPTION="Bulk Plasma Cells"
SUBSET_MODE="DICT"
SUBSET_MODE_CUSTOM_DICT=dict(lineage_group=dict(include=["P"], exclude=None))

# Table of Contents
<div id="toc"></div>

Setting up parameter block for papermill

Setting up output covariates and other things

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import decoupler as dc
from ipynbcompress import compress
import ipynbname
from datetime import datetime
import os


current_time = datetime.today().strftime("%Y-%m-%d;%H:%M:%S")


sub_directory = f'BULK_test/' 

# Inspect Object
output_dir_data_large = "/opt/megaseq-data/wcpilch/MMRF-ImmuneAtlas-Analysis-Sync/data/"
output_dir_fig_tables = "/opt/megaseq-data/wcpilch/MMRF-ImmuneAtlas-Analysis-Sync/output/RACE_ANALYSIS/" + sub_directory

os.makedirs(output_dir_fig_tables, exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
import gzip
from pathlib import Path

# Load gene list
G_list = pd.read_csv(os.path.join("/opt/pmedseq-data/wcpilch/Projects/MMRF-RaceAnalysis/MMRF_Emory_WP/output/Bulk_Race_Analyses", "grch38_and_grch37_ensembl_allids.csv"))

# Load count matrix
BULK_file = "/opt/megaseq-data/wcpilch/MMRF-ImmuneAtlas-Analysis-Sync/data/Bulk/COUNTS_Gene_Based_MMRF_CoMMpass_IA22_star_geneUnstranded_counts.tsv.gz"

# Read compressed TSV file with gene names as index
counts = pd.read_csv(BULK_file, sep='\t', index_col="Gene", compression='gzip')
counts = counts.iloc[:60623, :]  # remove qc lines
sample_list = list(counts.columns)

# Metadata file paths
visit_id_output_dir = "/opt/pmedseq-data/wcpilch/Projects/MMRF-ImmuneAtlas-Annotation/data/metadata_032525/all_commpass_per_visit/"
public_id_output_dir = "/opt/pmedseq-data/wcpilch/Projects/MMRF-ImmuneAtlas-Annotation/data/metadata_032525/all_commpass_per_pat/"
public_id_output_name = "032525_per_pat.rds"
visit_id_output_name = "032525_per_visit.rds"

# Load RDS files (assuming they're saved as pickle files or we need to use pyreadr)

import pyreadr
visitMD = pyreadr.read_r(os.path.join(visit_id_output_dir, visit_id_output_name))[None]
patMD = pyreadr.read_r(os.path.join(public_id_output_dir, public_id_output_name))[None]


# Extract sample information using string slicing
BM_or_PB = [x[12:14] for x in sample_list]  # Python uses 0-based indexing
pos_or_neg = [x[20:23] for x in sample_list]
visit_id = [x[0:11] for x in sample_list]
public_id = [x[0:9] for x in sample_list]

# Create bulk metadata dataframe
md_bulk = pd.DataFrame({
    'bulk_id': sample_list,
    'public_id': public_id,
    'd_visit_specimen_id': visit_id,
    'tissue_type': BM_or_PB
})

# Filter for BM tissue type
md_bulk = md_bulk[md_bulk['tissue_type'] == 'BM']

# Left join with patient and visit metadata
md_bulk = md_bulk.merge(patMD, on='public_id', how='left')
md_bulk = md_bulk.merge(visitMD, on=['public_id', 'd_visit_specimen_id'], how='left')
md_bulk = md_bulk.set_index('bulk_id')


# Filter for baseline samples with non-missing AFR ancestry
md_bulk_tmp = md_bulk[(md_bulk['VJ_INTERVAL'] == 'Baseline') & 
                      (md_bulk['ANCESTRY.AFR'].notna())]

# Select counts for final filtered samples
counts_final = counts[md_bulk_tmp.index]




In [ ]:
md_bulk_tmp.head()

In [ ]:
counts_final.head()

In [ ]:
# Create a scanpy AnnData object from counts_final and md_bulk_tmp
import scanpy as sc
import anndata as ad

# Create AnnData object
# counts_final should be genes x samples, so we need to transpose for scanpy (samples x genes)
adata = ad.AnnData(X=counts_final.T)

# Set gene names as var (variable/feature names)
adata.var_names = counts_final.index
adata.var_names_make_unique()

# Set sample names as obs_names (observation names)
adata.obs_names = counts_final.columns

# Add metadata from md_bulk_tmp to adata.obs
# Make sure the order matches
adata.obs = md_bulk_tmp.reindex(adata.obs_names)

# Display basic info about the AnnData object
print(f"AnnData object with n_obs × n_vars = {adata.n_obs} × {adata.n_vars}")
print(f"Sample metadata columns: {list(adata.obs.columns)}")
print(f"First 5 samples: {list(adata.obs_names[:5])}")
print(f"First 5 genes: {list(adata.var_names[:5])}")

adata

In [ ]:
# # Select counts for the filtered samples
# counts_select = counts[md_bulk.index]

# # Normalization and filtering (Python equivalent of edgeR operations)
# def calculate_cpm(counts_matrix, lib_sizes=None):
#     """Calculate counts per million (CPM)"""
#     if lib_sizes is None:
#         lib_sizes = counts_matrix.sum(axis=0)
    
#     # Calculate normalization factors (simplified TMM normalization)
#     norm_factors = lib_sizes / np.median(lib_sizes)
#     effective_lib_sizes = lib_sizes * norm_factors
    
#     # Calculate CPM
#     cpm_matrix = counts_matrix.div(effective_lib_sizes, axis=1) * 1e6
#     return cpm_matrix, norm_factors

# # Calculate CPM and normalization factors
# cpm_matrix, norm_factors = calculate_cpm(counts)

# # Filter low-expressed genes
# cutoff = 5
# max_cpm_per_gene = cpm_matrix.max(axis=1)
# genes_to_keep = max_cpm_per_gene >= cutoff

# # Apply filter
# counts_filtered = counts.loc[genes_to_keep]
# print(f"Number of genes after filtering: {counts_filtered.shape[0]}")


Printing time notebook was generated along with the container image version.

Note - the container image will only contain python analysis package and will not include notebook functionality or things like 'ipynbcompress'. These are installed in the devcontainer dockerfile.

<h1>Loading and initial preprocessing</h1>

Loading the full object, and checking file contents

In [ ]:
# Just checking embeddings
adata.obsm

In [ ]:
adata.obs.head()

Ensuring pandas is treating the continuous covariate columns we will be using as numeric covariates

In [ ]:
# Post conversion - numeric covariates treated as factors. Ensure they are correctly treated as numeric for downstream analyses.
cont_cov_col = ['d_dx_amm_age','d_dx_amm_bmi','ANCESTRY.AFR','ANCESTRY.PEL','ANCESTRY.EAS','ttcpfs','ttcos']

adata.obs[cont_cov_col] = adata.obs[cont_cov_col].apply(pd.to_numeric, errors='coerce')
adata.obs.dtypes

Subset to baseline + the cell types of interest

In [ ]:
steven_risk_calls_path = "/opt/megaseq-data/wcpilch/MMRF-ImmuneAtlas-Analysis-Sync/data/RacePaper/Steven_IA24_Risk_Calls"
steven_risk_calls = pd.read_csv(steven_risk_calls_path + "/CGS_risk_df.tsv", sep='\t')
steven_risk_calls.head()

In [ ]:
adata.obs = adata.obs.reset_index().merge(steven_risk_calls, on="public_id", how='left', suffixes=("","_FROM_TSV")).set_index('index')
adata.obs.head()

Creating other covariates and moving counts from raw -> layers

In [ ]:
# NOTE - FOR SCANPY VISUALIZATION PURPOSES ONLY. DOWNSTREAM ANALYSIS SHOULD ONLY UTILIZE THE ANCESTRY.AFR COVARIATE
adata.obs['AFR_BINNED'] = pd.cut(adata.obs['ANCESTRY.AFR'], bins=[-0.01, 0.5, 1.01], labels=['AFR Low', 'AFR High'])
adata.obs['AFR_BINNED'].value_counts()

In [ ]:
# Seurat creates a .raw slot instead of using .layers for counts data. Copy the raw matrix to a layer, and delete the raw slot.
adata.layers['counts'] = adata.X


<h1> Decoupler Pseudobulking </h1>

In [ ]:
#papermill_description=Pseudobulking
pdata = adata.copy()





In [ ]:
# Ensuring the covariates are correctly numeric in the pseudobulked object
cont_cov_col = ['d_dx_amm_age','d_dx_amm_bmi','ANCESTRY.AFR','ANCESTRY.PEL','ANCESTRY.EAS','ttcpfs','ttcos']

pdata.obs[cont_cov_col] = pdata.obs[cont_cov_col].apply(pd.to_numeric, errors='coerce')
pdata.obs.dtypes

Sample filtering 

Assessing for batch effect/variance in pseudobulked PCs

In [ ]:
# Store raw counts in layers
pdata.layers["counts"] = pdata.X.copy()

# Normalize, scale and compute pca
sc.pp.normalize_total(pdata, target_sum=1e4)
pdata.layers["nolog_normalized"] = pdata.X.copy()
sc.pp.log1p(pdata)
pdata.layers["normalized"] = pdata.X.copy()
sc.pp.scale(pdata, max_value=10)
pdata.layers["scaled"] = pdata.X.copy()

sc.tl.pca(pdata)

# Return raw counts to X
dc.pp.swap_layer(adata=pdata, key="counts", inplace=True)

In [ ]:
pdata.obs.head()

In [ ]:
pdata.obsm.keys()

In [ ]:
dc.tl.rankby_obsm(pdata, key="X_pca", obs_keys=["AFR_BINNED","d_pt_sex","d_dx_amm_age", "CGS_risk"])

In [ ]:
sc.pl.pca_variance_ratio(pdata)
dc.pl.obsm(adata=pdata, return_fig=True, nvar=4, titles=["PC scores", "Adjusted p-values"], figsize=(10, 5))


In [ ]:
sc.pl.pca(
    pdata,
    color=["AFR_BINNED", 'd_pt_sex', 'CGS_risk', 'd_dx_amm_age'],
    ncols=2,
    size=300,
    frameon=True,
)

Filtering by gene expression:

In [ ]:
dc.pl.filter_by_expr(
    adata=pdata,
    group="AFR_BINNED",
    min_count=10,
    min_total_count=15,
    large_n=10,
    min_prop=0.7,
)


In [ ]:
dc.pp.filter_by_expr(
    adata=pdata,
    group="AFR_BINNED",
    min_count=10,
    min_total_count=15,
    large_n=10,
    min_prop=0.7,
)


In [ ]:
pdata.obs.head()

In [ ]:
RELOAD_IF_EXISTING=False
FILE_NAME="Plasma_Bulk"


In [ ]:
pd.set_option('display.max_columns', None)
pdata.obs.head()

In [ ]:
# CONCISE SOLUTION: Convert all object columns to string in one line
# This is the most efficient approach for your use case

print("Original dtypes:")
print(pdata.obs.dtypes)
print()

# Convert all object columns to string
pdata.obs = pdata.obs.astype({col: 'category' for col in pdata.obs.select_dtypes(include=['object']).columns})

print("After converting object columns to string:")
print(pdata.obs.dtypes)

<h1> pyDESEQ Differential Expression Analysis </h1>

Using the pseudobulked profiles, uses deseq to find the relationship with ancestry, adjusting for other ancestry covariates, patient age, patient bmi, sex, scRNA processing center, and IMS/IMWG risk status. Assessing for significance of the ANCESTRY.AFR covariate. Saves out the DEG table with normal and unshrunk logFC values

In [ ]:
#papermill_description=pyDESEQ
# Import DESeq2
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats
import seaborn as sns 
import copy 

# Check if results already exist and should be reloaded. 
deg_results_file = output_dir_fig_tables + f"{FILE_NAME}_pydeseq2_diff_expression_lfc_shrink.csv"
if RELOAD_IF_EXISTING and os.path.exists(deg_results_file):
    print(f"Loading existing DEG results from: {deg_results_file}")
    merged_df = pd.read_csv(deg_results_file, index_col=0)
    
    # Still need to filter pdata for downstream analyses
    pdata_filt = pdata[pdata.obs['ANCESTRY.AFR'].notna() & pdata.obs['CGS_risk'].isin(['High', 'Standard']) & pdata.obs['d_dx_amm_age'].notna() & pdata.obs['d_dx_amm_bmi'].notna() & pdata.obs['d_pt_sex'].notna()].copy()
    pdata_filt.write_h5ad(output_dir_fig_tables + f"{FILE_NAME}_pydeseq2_filtered_pseudobulked.h5ad")

    print(f"Loaded {len(merged_df)} genes from existing results")
    print(merged_df.head())
else:
    print("Running DESeq2 analysis...")
    # Build DESeq2 object
    inference = DefaultInference(n_cpus=32)
    pdata_filt = pdata[pdata.obs['ANCESTRY.AFR'].notna() & pdata.obs['CGS_risk'].isin(['High', 'Standard']) & pdata.obs['d_dx_amm_age'].notna() & pdata.obs['d_dx_amm_bmi'].notna() & pdata.obs['d_pt_sex'].notna()].copy()
    # DESEQ adds a key that breaks saving the h5ad. Just moving the save out to here to avoid issues short term. Review for future projects
    pdata_filt.write_h5ad(output_dir_fig_tables + f"{FILE_NAME}_pydeseq2_filtered_pseudobulked.h5ad")
    pdata_filt.obs['AFR_BINNED'].value_counts()

    sample_meta = pdata_filt.obs[['public_id', 'AFR_BINNED', 'ANCESTRY.EUR', 'ANCESTRY.AFR', 'ANCESTRY.EAS', 'ANCESTRY.PEL', 'd_dx_amm_age', 'd_dx_amm_bmi', 'd_pt_sex', 'CGS_risk', 'censpfs', 'ttcpfs', 'censos', 'ttcos']]
    sample_meta.to_csv(output_dir_fig_tables + f"{FILE_NAME}_post_filter_sample_metadata.csv")

    model = "~ANCESTRY.AFR + ANCESTRY.EAS + ANCESTRY.PEL + d_dx_amm_age + d_dx_amm_bmi + d_pt_sex + CGS_risk"
    contrast = "ANCESTRY.AFR==1.0 - ANCESTRY.AFR==0.0"
    fdr_method = "BH (PyDESeq2 Default)"

    dds = DeseqDataSet(
        adata=pdata_filt,
        design=model,
        refit_cooks=True,
        inference=inference,
        quiet=True,
    )

    # Compute LFCs
    dds.deseq2()

    # Extract contrast between conditions
    stat_res = DeseqStats(dds, contrast=["ANCESTRY.AFR", 1.0, 0.0], inference=inference, quiet=True)

    # Compute Wald test
    print(stat_res.summary())
    stat_res_unshrunk_df = stat_res.results_df.copy()
    stat_res_shrunk = copy.deepcopy(stat_res) # Preserve orignal since shrinking is in-place

    stat_res_shrunk.lfc_shrink(coeff="ANCESTRY.AFR")

    # Comparing shink results

    stat_res_shrinked_df = stat_res_shrunk.results_df.copy()
    stat_res_shrinked_df = stat_res_shrinked_df.rename(columns={'log2FoldChange': 'log2FoldChange_lfcShrink', 'lfcSE': 'lfcSE_lfcShrink'})

    merged_df = pd.merge(stat_res_unshrunk_df, stat_res_shrinked_df[['log2FoldChange_lfcShrink', 'lfcSE_lfcShrink']], left_index=True, right_index=True, how='left')
    merged_df["model"] = model
    merged_df["contrast"] = contrast
    merged_df["FDR_method"] = fdr_method    
    merged_df = merged_df[['baseMean', 'log2FoldChange', 'lfcSE', 'log2FoldChange_lfcShrink', 'lfcSE_lfcShrink', 'stat', 'pvalue', 'padj', 'model', 'contrast', 'FDR_method', ]]
    gene_symbol_map = G_list.set_index('ensembl_gene_id')['hgnc_symbol_GRCh38']
    # Add the column using map
    merged_df['hgnc_symbol_GRCh38'] = merged_df.index.map(gene_symbol_map)

    print(merged_df.head())
    merged_df.to_csv(deg_results_file)

# pdata_filt.write_h5ad(output_dir_fig_tables + f"{FILE_NAME}_pydeseq2_filtered_pseudobulked.h5ad")

# Visualization
vis_df = merged_df[(merged_df['padj'].notna())].copy()
sns.scatterplot(
    vis_df,
    x='log2FoldChange',
    y='log2FoldChange_lfcShrink'
)



In [ ]:
# Sort merged_df by the column 'hgnc_symbol_GRCh38' in ascending order
merged_df_sorted = merged_df.sort_values(by='stat')

# For descending order:
merged_df_sorted_desc = merged_df.sort_values(by='stat', ascending=False)

merged_df_sorted.head(20)

In [ ]:
G_list[G_list["hgnc_symbol_GRCh38"] == "MXRA7"]

In [ ]:
merged_df[merged_df["hgnc_symbol_GRCh38"]=="MXRA7"]

In [ ]:
merged_df["alt_index"] = merged_df.index
merged_df

In [ ]:
# Prefix alt_index with hgnc_symbol_GRCh38 if not NaN

def prefix_alt_index(row):
    symbol = row['hgnc_symbol_GRCh38']
    alt_idx = row['alt_index']
    if pd.notnull(symbol):
        return f"{symbol}_{alt_idx}"
    else:
        return alt_idx

merged_df['alt_index_prefixed'] = merged_df.apply(prefix_alt_index, axis=1)

# Show a preview
print(merged_df[['alt_index', 'hgnc_symbol_GRCh38', 'alt_index_prefixed']].head(10))

<h2> Top Differentially expressed genes across the compartment of interest with respect to high (positive) or low (negative) African ancestry </h2>

Horizontal line indicates an adjusted p-value of 0.05. Vertical line indicates a log2Fold change of +/- 0.5

In [ ]:
# # Create a mapping from ensembl_gene_id to hgnc_symbol_GRCh38
# if 'ensembl_gene_id' in G_list.columns:
#     gene_symbol_map = G_list.set_index('ensembl_gene_id')['hgnc_symbol_GRCh38']
# else:
#     gene_symbol_map = G_list['hgnc_symbol_GRCh38']

# # Function to prefix symbol if available
# def prefix_symbol(ensembl_id):
#     symbol = gene_symbol_map.get(ensembl_id, None)
#     if pd.notnull(symbol):
#         return f"{symbol}_{ensembl_id}"
#     else:
#         return ensembl_id

# # Update var_names in pdata
# pdata.var_names = [prefix_symbol(eid) for eid in pdata.var_names]

# # Show a preview
# print(pdata.var_names[:10])

In [ ]:
dc.pl.volcano(merged_df, x="log2FoldChange", y="padj", top=30, figsize=(6, 5))

In [ ]:
merged_df

In [ ]:
tmp = merged_df.copy()
tmp = tmp[tmp["hgnc_symbol_GRCh38"].notnull()]
tmp = tmp.set_index('hgnc_symbol_GRCh38')
dc.pl.volcano(tmp, x="log2FoldChange", y="padj", top=30, figsize=(6, 5))


In [ ]:
merged_df = tmp.copy()

In [ ]:
merged_df
merged_df['Gene'] = merged_df.index

<h3> OmniPath Ligand-Receptor Analysis </h3>

In [ ]:
# Import OmniPath ligand-receptor interactions

import requests
import pandas as pd
import os
from datetime import datetime

# Set up cache directory and files
cache_dir = "data/py_omnipath"
cache_file = os.path.join(cache_dir, "lr_network.parquet")
log_file = os.path.join(cache_dir, "lr_network_download_log.txt")

# Create cache directory if it doesn't exist
os.makedirs(cache_dir, exist_ok=True)

# Check if cached file exists
if os.path.exists(cache_file):
    print(f"Loading lr_network from cache: {cache_file}")
    lr_network = pd.read_parquet(cache_file)
    
    # Read and display download date if log exists
    if os.path.exists(log_file):
        with open(log_file, 'r') as f:
            print(f"Cache info: {f.read().strip()}")
else:
    print("Cache not found. Fetching from OmniPath REST API...")
    
    url = "https://omnipathdb.org/interactions"
    params = {
        'datasets': 'ligrecextra',
        'fields': 'sources,references',
        'genesymbols': 'yes'
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        # Parse the response
        lines = response.text.strip().split('\n')
        header = lines[0].split('\t')
        data = [line.split('\t') for line in lines[1:]]
        
        lr_network = pd.DataFrame(data, columns=header)
        
        # Save to cache
        lr_network.to_parquet(cache_file, index=False)
        print(f"Saved lr_network to cache: {cache_file}")
        
        # Log download date
        download_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with open(log_file, 'w') as f:
            f.write(f"Downloaded on: {download_date}\n")
            f.write(f"Total interactions: {len(lr_network)}\n")
        print(f"Logged download date: {download_date}")
    else:
        print(f"Failed to fetch data: {response.status_code}")
        raise Exception(f"Failed to fetch OmniPath data: {response.status_code}")

# Process lr_network data
ligands = lr_network['source_genesymbol'].unique()
receptors = lr_network['target_genesymbol'].unique()

print(f"Found {len(ligands)} unique ligands")
print(f"Found {len(receptors)} unique receptors")
print(f"Total L-R interactions: {len(lr_network)}")

lr_mapping = lr_network[['source_genesymbol', 'target_genesymbol']].drop_duplicates()
lr_mapping.columns = ['ligand', 'receptor']

In [ ]:
# Subset pdata_filt to genes with non-NaN hgnc_symbol_GRCh38 and set var_names to hgnc_symbol_GRCh38
pdata_filt.var['hgnc_symbol_GRCh38'] = pdata_filt.var.index.map(gene_symbol_map)

# Ensure 'hgnc_symbol_GRCh38' exists in pdata_filt.var
if 'hgnc_symbol_GRCh38' not in pdata_filt.var.columns:
    raise ValueError("pdata_filt.var must contain a 'hgnc_symbol_GRCh38' column.")

# Get mask for genes with non-NaN hgnc_symbol_GRCh38
mask = pdata_filt.var['hgnc_symbol_GRCh38'].notna()

# Subset AnnData to those genes
pdata_filt = pdata_filt[:, mask].copy()

# Set var_names to hgnc_symbol_GRCh38
pdata_filt.var_names = pdata_filt.var['hgnc_symbol_GRCh38']

# Show preview
print(pdata_filt.var_names[:10])
print(f"Number of genes after filtering: {pdata_filt.n_vars}")

In [ ]:
# Subset differential expression results to ligands and receptors
deg_ligands = merged_df[merged_df.index.isin(ligands)].copy()
deg_receptors = merged_df[merged_df.index.isin(receptors)].copy()

print(f"\nDifferentially expressed genes:")
print(f"Ligands in DEG results: {len(deg_ligands)}")
print(f"Receptors in DEG results: {len(deg_receptors)}")

# Add significance flag
deg_ligands['significant'] = (deg_ligands['padj'] < 0.05)
deg_receptors['significant'] = (deg_receptors['padj'] < 0.05)

print(f"\nSignificant (padj < 0.05):")
print(f"Ligands: {deg_ligands['significant'].sum()}")
print(f"Receptors: {deg_receptors['significant'].sum()}")

In [ ]:
# Create volcano plot for ligands
fig, ax = plt.subplots(figsize=(10, 8))

# Plot non-significant points
non_sig = deg_ligands[~deg_ligands['significant']]
ax.scatter(non_sig['log2FoldChange'], -np.log10(non_sig['padj']), 
           c='gray', alpha=0.5, s=30, label='Non-significant')

# Plot significant points
sig = deg_ligands[deg_ligands['significant']]
ax.scatter(sig['log2FoldChange'], -np.log10(sig['padj']), 
           c='red', alpha=0.7, s=50, label='Significant (padj < 0.05)')

# Add labels for top significant genes
if len(sig) > 0:
    # Label top 10 by adjusted p-value
    top_genes = sig.nsmallest(100, 'padj')
    for gene_name, row in top_genes.iterrows():
        ax.annotate(gene_name, 
                   xy=(row['log2FoldChange'], -np.log10(row['padj'])),
                   xytext=(5, 5), textcoords='offset points',
                   fontsize=8, alpha=0.8)

# Add threshold lines
ax.axhline(y=-np.log10(0.05), color='blue', linestyle='--', linewidth=1, alpha=0.5, label='padj = 0.05')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)

ax.set_xlabel('log2 Fold Change (AFR High vs Low)', fontsize=12)
ax.set_ylabel('-log10(adjusted p-value)', fontsize=12)
ax.set_title('Volcano Plot: Differentially Expressed Ligands', fontsize=14, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Create volcano plot for receptors
fig, ax = plt.subplots(figsize=(10, 8))

# Plot non-significant points
non_sig = deg_receptors[~deg_receptors['significant']]
ax.scatter(non_sig['log2FoldChange'], -np.log10(non_sig['padj']), 
           c='gray', alpha=0.5, s=30, label='Non-significant')

# Plot significant points
sig = deg_receptors[deg_receptors['significant']]
ax.scatter(sig['log2FoldChange'], -np.log10(sig['padj']), 
           c='blue', alpha=0.7, s=50, label='Significant (padj < 0.05)')

# Add labels for top significant genes
if len(sig) > 0:
    # Label top 10 by adjusted p-value
    top_genes = sig.nsmallest(100, 'padj')
    for gene_name, row in top_genes.iterrows():
        ax.annotate(gene_name, 
                   xy=(row['log2FoldChange'], -np.log10(row['padj'])),
                   xytext=(5, 5), textcoords='offset points',
                   fontsize=8, alpha=0.8)

# Add threshold lines
ax.axhline(y=-np.log10(0.05), color='blue', linestyle='--', linewidth=1, alpha=0.5, label='padj = 0.05')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)

ax.set_xlabel('log2 Fold Change (AFR High vs Low)', fontsize=12)
ax.set_ylabel('-log10(adjusted p-value)', fontsize=12)
ax.set_title('Volcano Plot: Differentially Expressed Receptors', fontsize=14, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<h4> Significantly Differentially Expressed Ligands and Their Receptors </h4>

In [ ]:
# Get significant ligands and their receptors
sig_ligands = deg_ligands[deg_ligands['significant']].copy()

if len(sig_ligands) > 0:
    # For each significant ligand, find its receptors
    ligand_receptor_table = []
    
    for ligand in sig_ligands.index:
        # Get all receptors for this ligand
        receptors_for_ligand = lr_mapping[lr_mapping['ligand'] == ligand]['receptor'].tolist()
        
        # Get expression info for the ligand
        ligand_lfc = sig_ligands.loc[ligand, 'log2FoldChange']
        ligand_padj = sig_ligands.loc[ligand, 'padj']
        
        # Add row for each receptor
        if receptors_for_ligand:
            receptors_str = ', '.join(receptors_for_ligand[:10])  # Limit to first 10
            if len(receptors_for_ligand) > 10:
                receptors_str += f' ... (+{len(receptors_for_ligand)-10} more)'
            
            ligand_receptor_table.append({
                'Ligand': ligand,
                'log2FC': ligand_lfc,
                'padj': ligand_padj,
                'Known Receptors': receptors_str,
                'N_Receptors': len(receptors_for_ligand)
            })
    
    # Create DataFrame
    ligand_table = pd.DataFrame(ligand_receptor_table)
    ligand_table = ligand_table.sort_values('padj')
    
    print(f"Table of {len(ligand_table)} significantly differentially expressed ligands:\n")
    
    # Display with formatting
    from IPython.display import display
    display(ligand_table.style.format({
        'log2FC': '{:.3f}',
        'padj': '{:.2e}'
    }))
else:
    print("No significantly differentially expressed ligands found.")

<h4> Significantly Differentially Expressed Receptors and Their Ligands </h4>

In [ ]:
# Get significant receptors and their ligands
sig_receptors = deg_receptors[deg_receptors['significant']].copy()

if len(sig_receptors) > 0:
    # For each significant receptor, find its ligands
    receptor_ligand_table = []
    
    for receptor in sig_receptors.index:
        # Get all ligands for this receptor
        ligands_for_receptor = lr_mapping[lr_mapping['receptor'] == receptor]['ligand'].tolist()
        
        # Get expression info for the receptor
        receptor_lfc = sig_receptors.loc[receptor, 'log2FoldChange']
        receptor_padj = sig_receptors.loc[receptor, 'padj']
        
        # Add row for each ligand
        if ligands_for_receptor:
            ligands_str = ', '.join(ligands_for_receptor[:10])  # Limit to first 10
            if len(ligands_for_receptor) > 10:
                ligands_str += f' ... (+{len(ligands_for_receptor)-10} more)'
            
            receptor_ligand_table.append({
                'Receptor': receptor,
                'log2FC': receptor_lfc,
                'padj': receptor_padj,
                'Known Ligands': ligands_str,
                'N_Ligands': len(ligands_for_receptor)
            })
    
    # Create DataFrame
    receptor_table = pd.DataFrame(receptor_ligand_table)
    receptor_table = receptor_table.sort_values('padj')
    
    print(f"Table of {len(receptor_table)} significantly differentially expressed receptors:\n")
    
    # Display with formatting
    from IPython.display import display
    display(receptor_table.style.format({
        'log2FC': '{:.3f}',
        'padj': '{:.2e}'
    }))
else:
    print("No significantly differentially expressed receptors found.")

Fix for ULM names

In [ ]:
pdata_filt.obs['ANCESTRY_AFR'] = pdata_filt.obs['ANCESTRY.AFR']
pdata_filt.obs['ANCESTRY_EUR'] = pdata_filt.obs['ANCESTRY.EUR']
pdata_filt.obs['ANCESTRY_EAS'] = pdata_filt.obs['ANCESTRY.EAS']
pdata_filt.obs['ANCESTRY_PEL'] = pdata_filt.obs['ANCESTRY.PEL']

<h2> Decoupler Pathway/TF Analyses, based on differential expression results </h2>

All pathways are ranked such that positive corresponds to pathways enriched in 'AFR High', while negative corresponds to 'AFR Low'

In [ ]:
data = merged_df[["stat"]].T.rename(index={"stat": "AFRHigh_vs_AFRLow"})
data

<h3> Decoupler TF Analyses (collectri) </h3>

In [ ]:
#papermill_description=Decoupler_collectri
import decoupler as dc
import pickle
import os

# Define cache file path
cache_dir = "data/"
os.makedirs(cache_dir, exist_ok=True)
collectri_cache_file = os.path.join(cache_dir, "collectri_human.pkl")

# Check if cached version exists
if os.path.exists(collectri_cache_file):
    print(f"Loading collectri from cache: {collectri_cache_file}")
    with open(collectri_cache_file, 'rb') as f:
        collectri = pickle.load(f)
else:
    print("Downloading collectri from omnipath...")
    collectri = dc.op.collectri(organism="human")
    # Save to cache
    with open(collectri_cache_file, 'wb') as f:
        pickle.dump(collectri, f)
    print(f"Saved collectri to cache: {collectri_cache_file}")

collectri

In [ ]:
# Run
tf_acts, tf_padj = dc.mt.ulm(data=data, net=collectri)

if(len(tf_acts.columns) > 0):
    full_combined_df = pd.DataFrame({
        'pathway': tf_acts.columns,
        'score': tf_acts.values.flatten(),
        'padj': tf_padj[tf_acts.columns].values.flatten()
    })
else:
    full_combined_df = pd.DataFrame(columns=['pathway','score','padj'])

full_combined_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_decoupler_TF_regulon_score_unfiltered.tsv", sep='\t')


# Filter by sign padj
msk = (tf_padj.T < 0.05).iloc[:, 0]
tf_acts = tf_acts.loc[:, msk]

In [ ]:
# Combine re_acts and re_padj
if(len(tf_acts.columns) > 0):
    combined_df = pd.DataFrame({
        'pathway': tf_acts.columns,
        'score': tf_acts.values.flatten(),
        'padj': tf_padj[tf_acts.columns].values.flatten()
    })
else:
    combined_df = pd.DataFrame(columns=['pathway','score','padj'])

combined_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_decoupler_TF_regulon_score_fdr.tsv", sep='\t')
combined_df

In [ ]:
if(len(tf_acts.columns) > 0):
    dc.pl.barplot(data=tf_acts, name="AFRHigh_vs_AFRLow", figsize=(4, 12), top=100)
else:
    print("No significant TFs found")

In [ ]:
if(len(tf_acts.columns) > 0):
    dc.pl.network(
        net=collectri,
        data=data,
        score=tf_acts,
        sources=tf_acts.columns.tolist(),
        targets=10,
        figsize=(12, 7),
        vcenter=True,
        by_abs=True,
        size_node=30,
    )
else:
    print("No significant TFs found")

In [ ]:
if(len(tf_acts.columns) > 0):
    
    n_tfs = len(tf_acts.columns)
    if n_tfs > 1:
        n_cols = 4
        n_rows = (n_tfs + n_cols - 1) // n_cols  # Ceiling division
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
        axes = axes.flatten()
        
        for idx, tf in enumerate(tf_acts.columns.tolist()):
            dc.pl.volcano(
                data=merged_df,
                x="log2FoldChange",
                y="padj",
                net=collectri,
                name=tf,
                top=30,
                thr_stat=0.1,
                thr_sign=0.1,
                ax=axes[idx]
            )
            axes[idx].set_title(tf, fontsize=10)
        
        # Hide empty subplots
        for idx in range(n_tfs, len(axes)):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()
    elif(n_tfs==1):
        for tf in tf_acts.columns.tolist():
            dc.pl.volcano(
                data=merged_df,
                x="log2FoldChange",
                y="padj",
                net=collectri,
                name=tf,
                top=30,
                thr_stat=0.1,
                thr_sign=0.1
            )


else:
    print("No significant TFs found")


CollecTRI TF Alternative - Per-Sample ULM Analysis

In [ ]:
dc.mt.ulm(data=pdata_filt, net=collectri, layer='normalized')

In [ ]:
pdata_filt.obsm['score_ulm']

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm

# Your model from earlier
model_formula = "tf_score ~ 1 + ANCESTRY_AFR + ANCESTRY_EAS + ANCESTRY_PEL + d_dx_amm_age + d_dx_amm_bmi + d_pt_sex + CGS_risk"

# Fit for each TF (column in score_ulm)
results = {}
for tf in pdata_filt.obsm['score_ulm'].columns:
    # Add TF score to obs for modeling
    pdata_filt.obs['tf_score'] = pdata_filt.obsm['score_ulm'][tf]
    
    # Fit model using formula API (R-like syntax)
    model = smf.ols(model_formula, data=pdata_filt.obs).fit()
    
    # Extract coefficient and p-value for ANCESTRY_AFR
    coef = model.params['ANCESTRY_AFR']
    pval = model.pvalues['ANCESTRY_AFR']
    
    results[tf] = {
        'coefficient': coef,
        'p_value': pval,
        'std_err': model.bse['ANCESTRY_AFR'],
        't_stat': model.tvalues['ANCESTRY_AFR']
    }

In [ ]:
from statsmodels.stats.multitest import multipletests

results_df = pd.DataFrame.from_dict(results, orient='index')
results_df = results_df.sort_values(by='p_value')

# Apply Benjamini-Hochberg FDR correction
results_df['fdr_p_value'] = multipletests(results_df['p_value'], method='fdr_bh')[1]

# Save CollecTRI TF results to CSV
results_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_collectri_tf_ulm_regression_results.csv")

print(f"Saved CollecTRI TF regression results to: {output_dir_fig_tables}{FILE_NAME}_collectri_tf_ulm_regression_results.csv")

# Save CollecTRI TF ULM scores to CSV
collectri_ulm_scores = pd.DataFrame(pdata_filt.obsm['score_ulm'], index=pdata_filt.obs_names)
collectri_ulm_scores.to_csv(output_dir_fig_tables + f"{FILE_NAME}_collectri_tf_ulm_scores.csv")

print(f"Saved CollecTRI TF ULM scores to: {output_dir_fig_tables}{FILE_NAME}_collectri_tf_ulm_scores.csv")

results_df[results_df['p_value'] < 0.1]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Define color palettes
color_AFR_high = "#86325F"
color_AFR_low = "#5A90C7"
fill_AFR_high = "#9b4472"
fill_AFR_low = "#699ed4"

# Filter for significant TFs
sig_tfs = results_df[results_df['fdr_p_value'] < 0.05].sort_values(by='coefficient').index.tolist()

print(f"Found {len(sig_tfs)} significant TFs (FDR p < 0.05)")

if len(sig_tfs) > 0:
    n_tfs = len(sig_tfs)
    n_cols = 4
    n_rows = (n_tfs + n_cols - 1) // n_cols  # Ceiling division
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes = axes.flatten()
    
    for idx, tf in enumerate(sig_tfs):
        ax = axes[idx]
        
        # Get ULM scores for this TF
        tf_scores = pdata_filt.obsm['score_ulm'][tf]
        
        # Create a temporary dataframe for plotting
        plot_df = pd.DataFrame({
            'ULM_Score': tf_scores.values,
            'AFR_BINNED': pdata_filt.obs['AFR_BINNED'].values
        })
        
        # Reorder the categories so AFR High is on the right
        plot_df['AFR_BINNED'] = pd.Categorical(plot_df['AFR_BINNED'], 
                                                categories=['AFR Low', 'AFR High'], 
                                                ordered=True)
        
        # Get the p-value from results_df
        p_val = results_df.loc[tf, 'p_value']

        # Define color palettes for the two groups
        violin_colors = [fill_AFR_low, fill_AFR_high]
        box_colors = [color_AFR_low, color_AFR_high]
        
        # Plot violin plot with transparency
        parts = ax.violinplot([plot_df[plot_df['AFR_BINNED'] == 'AFR Low']['ULM_Score'].values,
                               plot_df[plot_df['AFR_BINNED'] == 'AFR High']['ULM_Score'].values],
                              positions=[0, 1],
                              showmeans=False, 
                              showmedians=False,
                              showextrema=False)
        
        # Color the violin plots with transparency
        for i, pc in enumerate(parts['bodies']):
            pc.set_facecolor(violin_colors[i])
            pc.set_alpha(0.4)
            pc.set_edgecolor('black')
            pc.set_linewidth(1)
        
        # Overlay box plot with custom colors
        bp = ax.boxplot([plot_df[plot_df['AFR_BINNED'] == 'AFR Low']['ULM_Score'].values,
                         plot_df[plot_df['AFR_BINNED'] == 'AFR High']['ULM_Score'].values],
                        positions=[0, 1],
                        widths=0.3,
                        patch_artist=True,
                        showfliers=True)
        
        # Color the box plots
        for i, (patch, color) in enumerate(zip(bp['boxes'], box_colors)):
            patch.set_facecolor(color)
            patch.set_alpha(0.8)
        
        # Color the other box plot elements
        for i, color in enumerate(box_colors):
            bp['medians'][i].set_color('black')
            bp['medians'][i].set_linewidth(2)
            bp['whiskers'][i*2].set_color(color)
            bp['whiskers'][i*2+1].set_color(color)
            bp['caps'][i*2].set_color(color)
            bp['caps'][i*2+1].set_color(color)
            if bp['fliers'][i].get_data()[0].size > 0:
                bp['fliers'][i].set_markerfacecolor(color)
                bp['fliers'][i].set_markeredgecolor(color)
                bp['fliers'][i].set_alpha(0.5)
        
        # Add individual points with matching colors
        for i, category in enumerate(['AFR Low', 'AFR High']):
            y_data = plot_df[plot_df['AFR_BINNED'] == category]['ULM_Score'].values
            x_data = np.random.normal(i, 0.04, size=len(y_data))
            ax.scatter(x_data, y_data, 
                      color=box_colors[i], 
                      alpha=0.3, 
                      s=30,
                      edgecolors='none')
        
        # Calculate position for p-value annotation and expand ylim
        y_max = plot_df['ULM_Score'].max()
        y_min = plot_df['ULM_Score'].min()
        y_range = y_max - y_min
        
        # Position p-value text 5% above the max value
        p_val_y = y_max + 0.05 * y_range
        
        # Expand y-axis to accommodate p-value text (add 12% above max to ensure text fits)
        ax.set_ylim(y_min - 0.05 * y_range, y_max + 0.15 * y_range)
        
        # Format p-value: 3 sig figs if >= 0.001, scientific notation if < 0.001
        if p_val >= 0.001:
            p_val_text = f'p = {p_val:.2g}'
        else:
            p_val_text = f'p = {p_val:.2e}'
        
        ax.text(0.5, p_val_y, 
               p_val_text, 
               ha='center', va='bottom', fontsize=12, fontweight='bold')
        
        # Set labels and title
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['AFR Low', 'AFR High'])
        ax.set_xlabel('African Genetic Similarity', fontsize=12)
        ax.set_ylabel('ULM Score', fontsize=12)
        ax.set_title(f'{tf}', fontsize=14, fontweight='bold')
        
    for idx in range(n_tfs, len(axes)):
        axes[idx].axis('off')    
    plt.tight_layout()
    plt.show()
else:
    print("No significant TFs to plot")

<h3> Progeny Pathway Analyses </h3>

In [ ]:
#papermill_description=Decoupler_progeny
import os
import pickle

# Cache progeny object to disk
progeny_cache_file = os.path.join(cache_dir, "progeny_human.pkl")
if os.path.exists(progeny_cache_file):
    print(f"Loading progeny from cache: {progeny_cache_file}")
    with open(progeny_cache_file, 'rb') as f:
        progeny = pickle.load(f)
else:
    print(f"Downloading progeny database...")
    progeny = dc.op.progeny(organism="human")
    print(f"Saving progeny to cache: {progeny_cache_file}")
    with open(progeny_cache_file, 'wb') as f:
        pickle.dump(progeny, f)

progeny

In [ ]:
# Run
pw_acts, pw_padj = dc.mt.ulm(data=data, net=progeny)

if(len(pw_acts.columns) > 0):
    full_combined_df = pd.DataFrame({
        'pathway': pw_acts.columns,
        'score': pw_acts.values.flatten(),
        'padj': pw_padj[pw_acts.columns].values.flatten()
    })
else:
    full_combined_df = pd.DataFrame(columns=['pathway','score','padj'])

full_combined_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_progeny_pathways_score_unfiltered.tsv", sep='\t')



# Filter by sign padj
msk = (pw_padj.T < 0.05).iloc[:, 0]
pw_acts = pw_acts.loc[:, msk]

pw_acts

In [ ]:
# Combine re_acts and re_padj
if(len(pw_acts.columns) > 0):
    combined_df = pd.DataFrame({
        'pathway': pw_acts.columns,
        'score': pw_acts.values.flatten(),
        'padj': pw_padj[pw_acts.columns].values.flatten()
    })
else:
    combined_df = pd.DataFrame(columns=['pathway','score','padj'])

In [ ]:
if(len(pw_acts.columns) > 0):
    dc.pl.barplot(data=pw_acts, name="AFRHigh_vs_AFRLow", figsize=(3, 3))
else:
    print("No significant progeny pathways found")

In [ ]:
if(len(pw_acts.columns) > 0):
    # Transform to df
    df = pw_acts.melt(value_name="score").merge(
        pw_padj.melt(value_name="padj")
        .assign(logpval=lambda x: x["padj"].clip(2.22e-4, 1))
        .assign(logpval=lambda x: -np.log10(x["logpval"]))
    )
    dc.pl.dotplot(df=df, x="score", y="variable", s="logpval", c="score", scale=1, figsize=(4, 4))
else:
    print("No significant progeny pathways found")

In [ ]:
#dc.pl.source_targets(data=merged_df, x="weight", y="stat", net=progeny, name="TNFa", top=15, figsize=(4, 4))

if(len(pw_acts.columns) > 0):
    
    n_pathways = len(pw_acts.columns)
    if n_pathways > 1:
        n_cols = 4
        n_rows = (n_pathways + n_cols - 1) // n_cols  # Ceiling division
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
        axes = axes.flatten()
        
        for idx, pathway in enumerate(pw_acts.columns.tolist()):
            print(pathway)
            dc.pl.source_targets(
                data=merged_df,
                x="weight",
                y="stat",
                net=progeny,
                name=pathway,
                top=20,
                ax=axes[idx]
            )
            axes[idx].set_title(pathway, fontsize=10)
        
        # Hide empty subplots
        for idx in range(n_pathways, len(axes)):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()
    elif(n_pathways==1):
        for pathway in pw_acts.columns.tolist():
            print(pathway)
            dc.pl.source_targets(
                data=merged_df,
                x="weight",
                y="stat",
                net=progeny,
                name=pathway,
                top=20,
                figsize=(4,4)
            )


else:
    print("No significant progeny pathways found")


In [ ]:
if(len(pw_acts.columns) > 0):
    pos_leading_edge_dict = {}
    neg_leading_edge_dict = {}
    
    n_pathways = len(pw_acts.columns)

    for pathway in pw_acts.columns.tolist():
        _, pos_le = dc.pl.leading_edge(
            merged_df,
            stat="stat",
            net=progeny[progeny["weight"] > 0],
            name=pathway,
        )
        pos_leading_edge_dict[pathway] = ', '.join(pos_le)
        print("(+) leading edge:", pos_le[:5])
        
        _, neg_le = dc.pl.leading_edge(
            merged_df,
            stat="stat",
            net=progeny[progeny["weight"] < 0],
            name=pathway,
        )
        print("(-) leading edge:", neg_le[:5])
        neg_leading_edge_dict[pathway] = ', '.join(neg_le)
    
    combined_df['POSITIVE_leading_edge_genes'] = combined_df['pathway'].map(pos_leading_edge_dict)
    combined_df['NEGATIVE_leading_edge_genes'] = combined_df['pathway'].map(neg_leading_edge_dict)

else:
    print("No significant progeny pathways found")


In [ ]:
combined_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_progeny_pathways_score_fdr_legenes.tsv", sep='\t')
combined_df

Progeny Pathway Alternative - Per-Sample ULM Analysis

In [ ]:
dc.mt.ulm(data=pdata_filt, net=progeny, layer='normalized')

In [ ]:
pdata_filt.obsm['score_ulm']

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm

# Your model from earlier
model_formula = "pathway_score ~ 1 + ANCESTRY_AFR + ANCESTRY_EAS + ANCESTRY_PEL + d_dx_amm_age + d_dx_amm_bmi + d_pt_sex + CGS_risk"

# Fit for each pathway (column in score_ulm)
results = {}
for pathway in pdata_filt.obsm['score_ulm'].columns:
    # Add pathway score to obs for modeling
    pdata_filt.obs['pathway_score'] = pdata_filt.obsm['score_ulm'][pathway]
    
    # Fit model using formula API (R-like syntax)
    model = smf.ols(model_formula, data=pdata_filt.obs).fit()
    
    # Extract coefficient and p-value for ANCESTRY_AFR
    coef = model.params['ANCESTRY_AFR']
    pval = model.pvalues['ANCESTRY_AFR']
    
    results[pathway] = {
        'coefficient': coef,
        'p_value': pval,
        'std_err': model.bse['ANCESTRY_AFR'],
        't_stat': model.tvalues['ANCESTRY_AFR']
    }

In [ ]:
from statsmodels.stats.multitest import multipletests

results_df = pd.DataFrame.from_dict(results, orient='index')
results_df = results_df.sort_values(by='p_value')

# Apply Benjamini-Hochberg FDR correction
results_df['fdr_p_value'] = multipletests(results_df['p_value'], method='fdr_bh')[1]

# Save Progeny pathway results to CSV
results_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_progeny_ulm_regression_results.csv")

print(f"Saved Progeny regression results to: {output_dir_fig_tables}{FILE_NAME}_progeny_ulm_regression_results.csv")

# Save Progeny ULM scores to CSV
progeny_ulm_scores = pd.DataFrame(pdata_filt.obsm['score_ulm'], index=pdata_filt.obs_names)
progeny_ulm_scores.to_csv(output_dir_fig_tables + f"{FILE_NAME}_progeny_ulm_scores.csv")

print(f"Saved Progeny ULM scores to: {output_dir_fig_tables}{FILE_NAME}_progeny_ulm_scores.csv")

results_df[results_df['p_value'] < 0.1]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Define color palettes
color_AFR_high = "#86325F"
color_AFR_low = "#5A90C7"
fill_AFR_high = "#9b4472"
fill_AFR_low = "#699ed4"

# Filter for significant pathways
sig_pathways = results_df[results_df['fdr_p_value'] < 0.05].sort_values(by='coefficient').index.tolist()

print(f"Found {len(sig_pathways)} significant pathways (FDR p < 0.05)")

if len(sig_pathways) > 0:
    n_pathways = len(sig_pathways)
    n_cols = 4
    n_rows = (n_pathways + n_cols - 1) // n_cols  # Ceiling division
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes = axes.flatten()
    
    for idx, pathway in enumerate(sig_pathways):
        ax = axes[idx]
        
        # Get ULM scores for this pathway
        pathway_scores = pdata_filt.obsm['score_ulm'][pathway]
        
        # Create a temporary dataframe for plotting
        plot_df = pd.DataFrame({
            'ULM_Score': pathway_scores.values,
            'AFR_BINNED': pdata_filt.obs['AFR_BINNED'].values
        })
        
        # Reorder the categories so AFR High is on the right
        plot_df['AFR_BINNED'] = pd.Categorical(plot_df['AFR_BINNED'], 
                                                categories=['AFR Low', 'AFR High'], 
                                                ordered=True)
        
        # Get the p-value from results_df
        p_val = results_df.loc[pathway, 'p_value']

        # Define color palettes for the two groups
        violin_colors = [fill_AFR_low, fill_AFR_high]
        box_colors = [color_AFR_low, color_AFR_high]
        
        # Plot violin plot with transparency
        parts = ax.violinplot([plot_df[plot_df['AFR_BINNED'] == 'AFR Low']['ULM_Score'].values,
                               plot_df[plot_df['AFR_BINNED'] == 'AFR High']['ULM_Score'].values],
                              positions=[0, 1],
                              showmeans=False, 
                              showmedians=False,
                              showextrema=False)
        
        # Color the violin plots with transparency
        for i, pc in enumerate(parts['bodies']):
            pc.set_facecolor(violin_colors[i])
            pc.set_alpha(0.4)
            pc.set_edgecolor('black')
            pc.set_linewidth(1)
        
        # Overlay box plot with custom colors
        bp = ax.boxplot([plot_df[plot_df['AFR_BINNED'] == 'AFR Low']['ULM_Score'].values,
                         plot_df[plot_df['AFR_BINNED'] == 'AFR High']['ULM_Score'].values],
                        positions=[0, 1],
                        widths=0.3,
                        patch_artist=True,
                        showfliers=True)
        
        # Color the box plots
        for i, (patch, color) in enumerate(zip(bp['boxes'], box_colors)):
            patch.set_facecolor(color)
            patch.set_alpha(0.8)
        
        # Color the other box plot elements
        for i, color in enumerate(box_colors):
            bp['medians'][i].set_color('black')
            bp['medians'][i].set_linewidth(2)
            bp['whiskers'][i*2].set_color(color)
            bp['whiskers'][i*2+1].set_color(color)
            bp['caps'][i*2].set_color(color)
            bp['caps'][i*2+1].set_color(color)
            if bp['fliers'][i].get_data()[0].size > 0:
                bp['fliers'][i].set_markerfacecolor(color)
                bp['fliers'][i].set_markeredgecolor(color)
                bp['fliers'][i].set_alpha(0.5)
        
        # Add individual points with matching colors
        for i, category in enumerate(['AFR Low', 'AFR High']):
            y_data = plot_df[plot_df['AFR_BINNED'] == category]['ULM_Score'].values
            x_data = np.random.normal(i, 0.04, size=len(y_data))
            ax.scatter(x_data, y_data, 
                      color=box_colors[i], 
                      alpha=0.3, 
                      s=30,
                      edgecolors='none')
        
        # Calculate position for p-value annotation and expand ylim
        y_max = plot_df['ULM_Score'].max()
        y_min = plot_df['ULM_Score'].min()
        y_range = y_max - y_min
        
        # Position p-value text 5% above the max value
        p_val_y = y_max + 0.05 * y_range
        
        # Expand y-axis to accommodate p-value text (add 12% above max to ensure text fits)
        ax.set_ylim(y_min - 0.05 * y_range, y_max + 0.15 * y_range)
        
        # Format p-value: 3 sig figs if >= 0.001, scientific notation if < 0.001
        if p_val >= 0.001:
            p_val_text = f'p = {p_val:.2g}'
        else:
            p_val_text = f'p = {p_val:.2e}'
        
        ax.text(0.5, p_val_y, 
               p_val_text, 
               ha='center', va='bottom', fontsize=12, fontweight='bold')
        
        # Set labels and title
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['AFR Low', 'AFR High'])
        ax.set_xlabel('African Genetic Similarity', fontsize=12)
        ax.set_ylabel('ULM Score', fontsize=12)
        ax.set_title(f'{pathway}', fontsize=14, fontweight='bold')
        
    for idx in range(n_pathways, len(axes)):
        axes[idx].axis('off')    
    plt.tight_layout()
    plt.show()
else:
    print("No significant pathways to plot")

<h3> Reactome Pathway Analysis </h3>

In [ ]:
#papermill_description=Decoupler_reactome
import gseapy as gp
import os
import pickle

msig = gp.Msigdb()
msig.list_dbver()
msig.list_category(dbver="2025.1.Hs")

# Cache reactome gmt object to disk
reactome_cache_file = os.path.join(cache_dir, "reactome_c2_cp_2025_1_Hs.pkl")
if os.path.exists(reactome_cache_file):
    print(f"Loading reactome from cache: {reactome_cache_file}")
    with open(reactome_cache_file, 'rb') as f:
        gmt = pickle.load(f)
else:
    print(f"Downloading reactome database...")
    gmt = msig.get_gmt(category="c2.cp.reactome", dbver="2025.1.Hs")
    print(f"Saving reactome to cache: {reactome_cache_file}")
    with open(reactome_cache_file, 'wb') as f:
        pickle.dump(gmt, f)

In [ ]:
# Convert CUSTOM_PATHWAYS dictionary to pandas DataFrame
# Each row is a gene-pathway pair
pathway_data_reactome = []
for pathway_name, genes in gmt.items():
    for gene in genes:
        pathway_data_reactome.append({'source': pathway_name, 'target': gene})

reactome_pathways_df = pd.DataFrame(pathway_data_reactome)
print(f"Created pathway dataframe with {len(reactome_pathways_df)} gene-pathway pairs")
print(f"Number of unique pathways: {reactome_pathways_df['source'].nunique()}")
print(f"Number of unique genes: {reactome_pathways_df['target'].nunique()}")
reactome_pathways_df = reactome_pathways_df.drop_duplicates(subset=['source','target'])
reactome_pathways_df.head()

In [ ]:
# Run
re_acts, re_padj = dc.mt.ulm(data=data, net=reactome_pathways_df)

if(len(re_acts.columns) > 0):
    full_combined_df = pd.DataFrame({
        'pathway': re_acts.columns,
        'score': re_acts.values.flatten(),
        'padj': re_padj[re_acts.columns].values.flatten()
    })
else:
    full_combined_df = pd.DataFrame(columns=['pathway','score','padj'])

full_combined_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_reactome_pathways_score_unfiltered.tsv", sep='\t')


# Filter by sign padj
msk = (re_padj.T < 0.05).iloc[:, 0]
re_acts = re_acts.loc[:, msk]

re_acts

In [ ]:
# Combine re_acts and re_padj
if(len(re_acts.columns) > 0):
    combined_df = pd.DataFrame({
        'pathway': re_acts.columns,
        'score': re_acts.values.flatten(),
        'padj': re_padj[re_acts.columns].values.flatten()
    })
else:
    combined_df = pd.DataFrame(columns=['pathway','score','padj'])


In [ ]:
if(len(re_acts.columns) > 0):
    dc.pl.barplot(data=re_acts, name="AFRHigh_vs_AFRLow", figsize=(10, min(len(re_acts.columns) * 3/5, 50*3/5)), top=50)
else:
    print("No significant reactome pathways found")

In [ ]:
# Add leading edge genes to combined_df
if len(re_acts.columns) > 0:
    leading_edge_dict = {}
    
    n_pathways = len(re_acts.columns)

    for pathway in re_acts.columns.tolist():
        _, le = dc.pl.leading_edge(
            merged_df,
            stat="stat",
            net=reactome_pathways_df,
            name=pathway,
        )
        leading_edge_dict[pathway] = ', '.join(le)
    
    # Add as a new column
    combined_df['leading_edge_genes'] = combined_df['pathway'].map(leading_edge_dict)
else:
    print("No significant reactome pathways found")


In [ ]:
combined_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_reactome_pathways_score_fdr_legenes.tsv", sep='\t')
combined_df

In [ ]:
if(len(re_acts.columns) > 0):
    
    n_pathways = len(re_acts.columns)
    if n_pathways > 1:
        n_cols = 4
        n_rows = (n_pathways + n_cols - 1) // n_cols  # Ceiling division
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
        axes = axes.flatten()
        
        for idx, pathway in enumerate(re_acts.columns.tolist()):
            dc.pl.volcano(
                data=merged_df,
                x="log2FoldChange",
                y="padj",
                net=reactome_pathways_df,
                name=pathway,
                top=30,
                thr_stat=0.1,
                thr_sign=0.1,
                ax=axes[idx]
            )
            axes[idx].set_title(pathway, fontsize=10)
        
        # Hide empty subplots
        for idx in range(n_pathways, len(axes)):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()
    elif(n_pathways==1):
        for pathway in re_acts.columns.tolist():
            dc.pl.volcano(
                data=merged_df,
                x="log2FoldChange",
                y="padj",
                net=reactome_pathways_df,
                name=pathway,
                top=30,
                thr_stat=0.1,
                thr_sign=0.1
            )


else:
    print("No significant reactome pathways found")

In [ ]:
dc.mt.ulm(data=pdata_filt, net=reactome_pathways_df, layer='normalized')

In [ ]:
pdata_filt.obsm['score_ulm']

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm

# Your model from earlier
model_formula = "pathway_score ~ 1 + ANCESTRY_AFR + ANCESTRY_EAS + ANCESTRY_PEL + d_dx_amm_age + d_dx_amm_bmi + d_pt_sex + CGS_risk"

# Fit for each pathway (column in score_ulm)
results = {}
for pathway in reactome_pathways_df['source'].unique():
    if pathway in pdata_filt.obsm['score_ulm'].columns:
        # Add pathway score to obs for modeling
        pdata_filt.obs['pathway_score'] = pdata_filt.obsm['score_ulm'][pathway]
        
        # Fit model using formula API (R-like syntax)
        model = smf.ols(model_formula, data=pdata_filt.obs).fit()
        
        # Extract coefficient and p-value for ANCESTRY_AFR
        coef = model.params['ANCESTRY_AFR']
        pval = model.pvalues['ANCESTRY_AFR']
        
        results[pathway] = {
            'coefficient': coef,
            'p_value': pval,
            'std_err': model.bse['ANCESTRY_AFR'],
            't_stat': model.tvalues['ANCESTRY_AFR']
        }

In [ ]:
from statsmodels.stats.multitest import multipletests

results_df = pd.DataFrame.from_dict(results, orient='index')
results_df = results_df.sort_values(by='p_value')

# Apply Benjamini-Hochberg FDR correction
results_df['fdr_p_value'] = multipletests(results_df['p_value'], method='fdr_bh')[1]
# Save hallmark results to CSV

results_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_reactome_ulm_regression_results.csv")

print(f"Saved reactome regression results to: {output_dir_fig_tables}{FILE_NAME}_reactome_ulm_regression_results.csv")

print(f"Saved reactome ULM scores to: {output_dir_fig_tables}{FILE_NAME}_reactome_ulm_scores.csv")
# Save reactome ULM scores to CSV
reactome_ulm_scores = pd.DataFrame(pdata_filt.obsm['score_ulm'], index=pdata_filt.obs_names)
reactome_ulm_scores.to_csv(output_dir_fig_tables + f"{FILE_NAME}_reactome_ulm_scores.csv")

results_df[results_df['p_value'] < 0.1]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Define color palettes
color_AFR_high = "#86325F"
color_AFR_low = "#5A90C7"
fill_AFR_high = "#9b4472"
fill_AFR_low = "#699ed4"

# Filter for significant pathways
sig_pathways = results_df[results_df['fdr_p_value'] < 0.05].sort_values(by='coefficient').index.tolist()

print(f"Found {len(sig_pathways)} significant pathways (p < 0.05)")

if len(sig_pathways) > 0:
    n_pathways = len(sig_pathways)
    n_cols = 4
    n_rows = (n_pathways + n_cols - 1) // n_cols  # Ceiling division
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes = axes.flatten()
    
    for idx, pathway in enumerate(sig_pathways):
        ax = axes[idx]
        
        # Get ULM scores for this pathway
        pathway_scores = pdata_filt.obsm['score_ulm'][pathway]
        
        # Create a temporary dataframe for plotting
        plot_df = pd.DataFrame({
            'ULM_Score': pathway_scores.values,
            'AFR_BINNED': pdata_filt.obs['AFR_BINNED'].values
        })
        
        # Reorder the categories so AFR High is on the right
        plot_df['AFR_BINNED'] = pd.Categorical(plot_df['AFR_BINNED'], 
                                                categories=['AFR Low', 'AFR High'], 
                                                ordered=True)
        
        # Get the p-value from results_df
        p_val = results_df.loc[pathway, 'p_value']

        # Define color palettes for the two groups
        violin_colors = [fill_AFR_low, fill_AFR_high]
        box_colors = [color_AFR_low, color_AFR_high]
        
        # Plot violin plot with transparency
        parts = ax.violinplot([plot_df[plot_df['AFR_BINNED'] == 'AFR Low']['ULM_Score'].values,
                               plot_df[plot_df['AFR_BINNED'] == 'AFR High']['ULM_Score'].values],
                              positions=[0, 1],
                              showmeans=False, 
                              showmedians=False,
                              showextrema=False)
        
        # Color the violin plots with transparency
        for i, pc in enumerate(parts['bodies']):
            pc.set_facecolor(violin_colors[i])
            pc.set_alpha(0.4)
            pc.set_edgecolor('black')
            pc.set_linewidth(1)
        
        # Overlay box plot with custom colors
        bp = ax.boxplot([plot_df[plot_df['AFR_BINNED'] == 'AFR Low']['ULM_Score'].values,
                         plot_df[plot_df['AFR_BINNED'] == 'AFR High']['ULM_Score'].values],
                        positions=[0, 1],
                        widths=0.3,
                        patch_artist=True,
                        showfliers=True)
        
        # Color the box plots
        for i, (patch, color) in enumerate(zip(bp['boxes'], box_colors)):
            patch.set_facecolor(color)
            patch.set_alpha(0.8)
        
        # Color the other box plot elements
        for i, color in enumerate(box_colors):
            bp['medians'][i].set_color('black')
            bp['medians'][i].set_linewidth(2)
            bp['whiskers'][i*2].set_color(color)
            bp['whiskers'][i*2+1].set_color(color)
            bp['caps'][i*2].set_color(color)
            bp['caps'][i*2+1].set_color(color)
            if bp['fliers'][i].get_data()[0].size > 0:
                bp['fliers'][i].set_markerfacecolor(color)
                bp['fliers'][i].set_markeredgecolor(color)
                bp['fliers'][i].set_alpha(0.5)
        
        # Add individual points with matching colors
        for i, category in enumerate(['AFR Low', 'AFR High']):
            y_data = plot_df[plot_df['AFR_BINNED'] == category]['ULM_Score'].values
            x_data = np.random.normal(i, 0.04, size=len(y_data))
            ax.scatter(x_data, y_data, 
                      color=box_colors[i], 
                      alpha=0.3, 
                      s=30,
                      edgecolors='none')
        
        # Calculate position for p-value annotation and expand ylim
        y_max = plot_df['ULM_Score'].max()
        y_min = plot_df['ULM_Score'].min()
        y_range = y_max - y_min
        
        # Position p-value text 5% above the max value
        p_val_y = y_max + 0.05 * y_range
        
        # Expand y-axis to accommodate p-value text (add 12% above max to ensure text fits)
        ax.set_ylim(y_min - 0.05 * y_range, y_max + 0.15 * y_range)
        
        # Format p-value: 3 sig figs if >= 0.001, scientific notation if < 0.001
        if p_val >= 0.001:
            p_val_text = f'p = {p_val:.2g}'
        else:
            p_val_text = f'p = {p_val:.2e}'
        
        ax.text(0.5, p_val_y, 
               p_val_text, 
               ha='center', va='bottom', fontsize=12, fontweight='bold')
        
        # Set labels and title
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['AFR Low', 'AFR High'])
        ax.set_xlabel('African Genetic Similarity', fontsize=12)
        ax.set_ylabel('ULM Score', fontsize=12)
        ax.set_title(f'{pathway}', fontsize=14, fontweight='bold')
        
    for idx in range(n_pathways, len(axes)):
        axes[idx].axis('off')    
    plt.tight_layout()
    plt.show()
else:
    print("No significant pathways to plot")

<h3> Hallmark Pathway Analyses </h3>

In [ ]:
#papermill_description=Decoupler_hallmark
import os
import pickle

# Cache hallmark object to disk
hallmark_cache_file = os.path.join(cache_dir, "hallmark_human.pkl")
if os.path.exists(hallmark_cache_file):
    print(f"Loading hallmark from cache: {hallmark_cache_file}")
    with open(hallmark_cache_file, 'rb') as f:
        hallmark = pickle.load(f)
else:
    print(f"Downloading hallmark database...")
    hallmark = dc.op.hallmark(organism="human")
    print(f"Saving hallmark to cache: {hallmark_cache_file}")
    with open(hallmark_cache_file, 'wb') as f:
        pickle.dump(hallmark, f)

hallmark

In [ ]:
# Run
hm_acts, hm_padj = dc.mt.ulm(data=data, net=hallmark)

if(len(hm_acts.columns) > 0):
    full_combined_df = pd.DataFrame({
        'pathway': hm_acts.columns,
        'score': hm_acts.values.flatten(),
        'padj': hm_padj[hm_acts.columns].values.flatten()
    })
else:
    full_combined_df = pd.DataFrame(columns=['pathway','score','padj'])

full_combined_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_hallmark_pathways_score_unfiltered.tsv", sep='\t')


# Filter by sign padj
msk = (hm_padj.T < 0.05).iloc[:, 0]
hm_acts = hm_acts.loc[:, msk]

hm_acts

In [ ]:
if(len(hm_acts.columns) > 0):
    # Combine hm_acts and hm_padj
    combined_df = pd.DataFrame({
        'pathway': hm_acts.columns,
        'score': hm_acts.values.flatten(),
        'padj': hm_padj[hm_acts.columns].values.flatten()
    })
else:
    combined_df = pd.DataFrame(columns=['pathway','score','padj'])


In [ ]:
if(len(hm_acts.columns) > 0):
    dc.pl.barplot(data=hm_acts, name="AFRHigh_vs_AFRLow", figsize=(10, min(len(hm_acts.columns) * 3/5, 50*3/5)), top=50)
else:
    print("No significant hallmark pathways found")

In [ ]:
if(len(hm_acts.columns) > 0):
    leading_edge_dict = {}
    
    n_pathways = len(hm_acts.columns)

    for pathway in hm_acts.columns.tolist():
        _, le = dc.pl.leading_edge(
            merged_df,
            stat="stat",
            net=hallmark,
            name=pathway,
        )
        leading_edge_dict[pathway] = ', '.join(le)
        print(f"{pathway} leading edge:", le[:10])
    
    combined_df['leading_edge_genes'] = combined_df['pathway'].map(leading_edge_dict)
else:
    print("No significant hallmark pathways found")


In [ ]:
combined_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_hallmark_pathways_score_fdr_legenes.tsv", sep='\t')
combined_df

In [ ]:
if(len(hm_acts.columns) > 0):
    
    n_pathways = len(hm_acts.columns)
    if n_pathways > 1:
        n_cols = 4
        n_rows = (n_pathways + n_cols - 1) // n_cols  # Ceiling division
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
        axes = axes.flatten()
        
        for idx, pathway in enumerate(hm_acts.columns.tolist()):
            dc.pl.volcano(
                data=merged_df,
                x="log2FoldChange",
                y="padj",
                net=hallmark,
                name=pathway,
                top=30,
                thr_stat=0.1,
                thr_sign=0.1,
                ax=axes[idx]
            )
            axes[idx].set_title(pathway, fontsize=10)
        
        # Hide empty subplots
        for idx in range(n_pathways, len(axes)):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()
    elif(n_pathways==1):
        for pathway in hm_acts.columns.tolist():
            dc.pl.volcano(
                data=merged_df,
                x="log2FoldChange",
                y="padj",
                net=hallmark,
                name=pathway,
                top=30,
                thr_stat=0.1,
                thr_sign=0.1
            )


else:
    print("No significant hallmark pathways found")

HALLMARK Alternative - ULM Visualization

In [ ]:
dc.mt.ulm(data=pdata_filt, net=hallmark, layer='normalized')

In [ ]:
pdata_filt.obsm['score_ulm']

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm

# Your model from earlier
model_formula = "pathway_score ~ 1 + ANCESTRY_AFR + ANCESTRY_EAS + ANCESTRY_PEL + d_dx_amm_age + d_dx_amm_bmi + d_pt_sex + CGS_risk"

# Fit for each pathway (column in score_ulm)
results = {}
for pathway in pdata_filt.obsm['score_ulm'].columns:
    # Add pathway score to obs for modeling
    pdata_filt.obs['pathway_score'] = pdata_filt.obsm['score_ulm'][pathway]
    
    # Fit model using formula API (R-like syntax)
    model = smf.ols(model_formula, data=pdata_filt.obs).fit()
    
    # Extract coefficient and p-value for ANCESTRY_AFR
    coef = model.params['ANCESTRY_AFR']
    pval = model.pvalues['ANCESTRY_AFR']
    
    results[pathway] = {
        'coefficient': coef,
        'p_value': pval,
        'std_err': model.bse['ANCESTRY_AFR'],
        't_stat': model.tvalues['ANCESTRY_AFR']
    }

In [ ]:
from statsmodels.stats.multitest import multipletests

results_df = pd.DataFrame.from_dict(results, orient='index')
results_df = results_df.sort_values(by='p_value')

# Apply Benjamini-Hochberg FDR correction
results_df['fdr_p_value'] = multipletests(results_df['p_value'], method='fdr_bh')[1]
# Save hallmark results to CSV

results_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_hallmark_ulm_regression_results.csv")

print(f"Saved hallmark regression results to: {output_dir_fig_tables}{FILE_NAME}_hallmark_ulm_regression_results.csv")

print(f"Saved hallmark ULM scores to: {output_dir_fig_tables}{FILE_NAME}_hallmark_ulm_scores.csv")

# Save hallmark ULM scores to CSVhallmark_ulm_scores.to_csv(output_dir_fig_tables + f"{FILE_NAME}_hallmark_ulm_scores.csv")
hallmark_ulm_scores = pd.DataFrame(pdata_filt.obsm['score_ulm'], index=pdata_filt.obs_names)
hallmark_ulm_scores.to_csv(output_dir_fig_tables + f"{FILE_NAME}_hallmark_ulm_scores.csv")

results_df[results_df['p_value'] < 0.1]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Define color palettes
color_AFR_high = "#86325F"
color_AFR_low = "#5A90C7"
fill_AFR_high = "#9b4472"
fill_AFR_low = "#699ed4"

# Filter for significant pathways
sig_pathways = results_df[results_df['fdr_p_value'] < 0.05].sort_values(by='coefficient').index.tolist()

print(f"Found {len(sig_pathways)} significant pathways (p < 0.05)")

if len(sig_pathways) > 0:
    n_pathways = len(sig_pathways)
    n_cols = 4
    n_rows = (n_pathways + n_cols - 1) // n_cols  # Ceiling division
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes = axes.flatten() if n_pathways > 1 else [axes]
    
    for idx, pathway in enumerate(sig_pathways):
        ax = axes[idx]
        
        # Get ULM scores for this pathway
        pathway_scores = pdata_filt.obsm['score_ulm'][pathway]
        
        # Create a temporary dataframe for plotting
        plot_df = pd.DataFrame({
            'ULM_Score': pathway_scores.values,
            'AFR_BINNED': pdata_filt.obs['AFR_BINNED'].values
        })
        
        # Reorder the categories so AFR High is on the right
        plot_df['AFR_BINNED'] = pd.Categorical(plot_df['AFR_BINNED'], 
                                                categories=['AFR Low', 'AFR High'], 
                                                ordered=True)
        
        # Get the p-value from results_df
        p_val = results_df.loc[pathway, 'p_value']
        
        # Define color palettes for the two groups
        violin_colors = [fill_AFR_low, fill_AFR_high]
        box_colors = [color_AFR_low, color_AFR_high]
        
        # Plot violin plot with transparency
        parts = ax.violinplot([plot_df[plot_df['AFR_BINNED'] == 'AFR Low']['ULM_Score'].values,
                               plot_df[plot_df['AFR_BINNED'] == 'AFR High']['ULM_Score'].values],
                              positions=[0, 1],
                              showmeans=False, 
                              showmedians=False,
                              showextrema=False)
        
        # Color the violin plots with transparency
        for i, pc in enumerate(parts['bodies']):
            pc.set_facecolor(violin_colors[i])
            pc.set_alpha(0.4)
            pc.set_edgecolor('black')
            pc.set_linewidth(1)
        
        # Overlay box plot with custom colors
        bp = ax.boxplot([plot_df[plot_df['AFR_BINNED'] == 'AFR Low']['ULM_Score'].values,
                         plot_df[plot_df['AFR_BINNED'] == 'AFR High']['ULM_Score'].values],
                        positions=[0, 1],
                        widths=0.3,
                        patch_artist=True,
                        showfliers=True)
        
        # Color the box plots
        for i, (patch, color) in enumerate(zip(bp['boxes'], box_colors)):
            patch.set_facecolor(color)
            patch.set_alpha(0.8)
        
        # Color the other box plot elements
        for i, color in enumerate(box_colors):
            bp['medians'][i].set_color('black')
            bp['medians'][i].set_linewidth(2)
            bp['whiskers'][i*2].set_color(color)
            bp['whiskers'][i*2+1].set_color(color)
            bp['caps'][i*2].set_color(color)
            bp['caps'][i*2+1].set_color(color)
            if bp['fliers'][i].get_data()[0].size > 0:
                bp['fliers'][i].set_markerfacecolor(color)
                bp['fliers'][i].set_markeredgecolor(color)
                bp['fliers'][i].set_alpha(0.5)
        
        # Add individual points with matching colors
        for i, category in enumerate(['AFR Low', 'AFR High']):
            y_data = plot_df[plot_df['AFR_BINNED'] == category]['ULM_Score'].values
            x_data = np.random.normal(i, 0.04, size=len(y_data))
            ax.scatter(x_data, y_data, 
                      color=box_colors[i], 
                      alpha=0.3, 
                      s=30,
                      edgecolors='none')
        
        # Calculate position for p-value annotation and expand ylim
        y_max = plot_df['ULM_Score'].max()
        y_min = plot_df['ULM_Score'].min()
        y_range = y_max - y_min
        
        # Position p-value text 5% above the max value
        p_val_y = y_max + 0.05 * y_range
        
        # Expand y-axis to accommodate p-value text (add 12% above max to ensure text fits)
        ax.set_ylim(y_min - 0.05 * y_range, y_max + 0.15 * y_range)
        
        # Format p-value: 3 sig figs if >= 0.001, scientific notation if < 0.001
        if p_val >= 0.001:
            p_val_text = f'p = {p_val:.2g}'
        else:
            p_val_text = f'p = {p_val:.2e}'
        
        ax.text(0.5, p_val_y, 
               p_val_text, 
               ha='center', va='bottom', fontsize=12, fontweight='bold')
        
        # Set labels and title
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['AFR Low', 'AFR High'])
        ax.set_xlabel('African Genetic Similarity', fontsize=12)
        ax.set_ylabel('ULM Score', fontsize=12)
        ax.set_title(f'{pathway}', fontsize=14, fontweight='bold')
              # Hide empty subplots
    for idx in range(n_pathways, len(axes)):
        axes[idx].axis('off')
        
    plt.tight_layout()
    plt.show()
else:
    print("No significant pathways to plot")

<h3> GO Pathway Analysis </h3>

In [ ]:
#papermill_description=Decoupler_go
import os
import pickle

# Cache GO:BP gmt object to disk
gobp_cache_file = os.path.join(cache_dir, "go_bp_c5_2025_1_Hs.pkl")
if os.path.exists(gobp_cache_file):
    print(f"Loading GO:BP from cache: {gobp_cache_file}")
    with open(gobp_cache_file, 'rb') as f:
        gmt = pickle.load(f)
else:
    print(f"Downloading GO:BP database...")
    gmt = msig.get_gmt(category="c5.go.bp", dbver="2025.1.Hs")
    print(f"Saving GO:BP to cache: {gobp_cache_file}")
    with open(gobp_cache_file, 'wb') as f:
        pickle.dump(gmt, f)

In [ ]:
# Convert CUSTOM_PATHWAYS dictionary to pandas DataFrame
# Each row is a gene-pathway pair
pathway_data_gobp = []
for pathway_name, genes in gmt.items():
    for gene in genes:
        pathway_data_gobp.append({'source': pathway_name, 'target': gene})

gobp_pathways_df = pd.DataFrame(pathway_data_gobp)
print(f"Created pathway dataframe with {len(gobp_pathways_df)} gene-pathway pairs")
print(f"Number of unique pathways: {gobp_pathways_df['source'].nunique()}")
print(f"Number of unique genes: {gobp_pathways_df['target'].nunique()}")
gobp_pathways_df = gobp_pathways_df.drop_duplicates(subset=['source','target'])
gobp_pathways_df.head()

In [ ]:
# Run
gobp_acts, gobp_padj = dc.mt.ulm(data=data, net=gobp_pathways_df)

if(len(gobp_acts.columns) > 0):
    full_combined_df = pd.DataFrame({
        'pathway': gobp_acts.columns,
        'score': gobp_acts.values.flatten(),
        'padj': gobp_padj[gobp_acts.columns].values.flatten()
    })
else:
    full_combined_df = pd.DataFrame(columns=['pathway','score','padj'])

full_combined_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_gobp_pathways_score_unfiltered.tsv", sep='\t')


# Filter by sign padj
msk = (gobp_padj.T < 0.05).iloc[:, 0]
gobp_acts = gobp_acts.loc[:, msk]

gobp_acts

In [ ]:
# Combine gobp_acts and gobp_padj
if(len(gobp_acts.columns) > 0):
    combined_df = pd.DataFrame({
        'pathway': gobp_acts.columns,
        'score': gobp_acts.values.flatten(),
        'padj': gobp_padj[gobp_acts.columns].values.flatten()
    })
else:
    combined_df = pd.DataFrame(columns=['pathway','score','padj'])


In [ ]:
if(len(gobp_acts.columns) > 0):
    dc.pl.barplot(data=gobp_acts, name="AFRHigh_vs_AFRLow", figsize=(10, min(len(gobp_acts.columns) * 3/5, 50*3/5)), top=50)
else:
    print("No significant C5-GO:BP pathways found")

In [ ]:
# Add leading edge genes to combined_df
if len(gobp_acts.columns) > 0:
    leading_edge_dict = {}
    
    n_pathways = len(gobp_acts.columns)

    for pathway in gobp_acts.columns.tolist():
        _, le = dc.pl.leading_edge(
            merged_df,
            stat="stat",
            net=gobp_pathways_df,
            name=pathway,
        )
        leading_edge_dict[pathway] = ', '.join(le)
    
    # Add as a new column
    combined_df['leading_edge_genes'] = combined_df['pathway'].map(leading_edge_dict)
else:
    print("No significant C5-GO:BP pathways found")


In [ ]:
combined_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_gobp_pathways_score_fdr_legenes.tsv", sep='\t')
combined_df

In [ ]:
if(len(gobp_acts.columns) > 0):
    
    n_pathways = len(gobp_acts.columns)
    if n_pathways > 1:
        n_cols = 4
        n_rows = (n_pathways + n_cols - 1) // n_cols  # Ceiling division
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
        axes = axes.flatten()
        
        for idx, pathway in enumerate(gobp_acts.columns.tolist()):
            dc.pl.volcano(
                data=merged_df,
                x="log2FoldChange",
                y="padj",
                net=gobp_pathways_df,
                name=pathway,
                top=30,
                thr_stat=0.1,
                thr_sign=0.1,
                ax=axes[idx]
            )
            axes[idx].set_title(pathway, fontsize=10)
        
        # Hide empty subplots
        for idx in range(n_pathways, len(axes)):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()
    elif(n_pathways==1):
        for pathway in re_acts.columns.tolist():
            dc.pl.volcano(
                data=merged_df,
                x="log2FoldChange",
                y="padj",
                net=gobp_pathways_df,
                name=pathway,
                top=30,
                thr_stat=0.1,
                thr_sign=0.1
            )


else:
    print("No significant C5-GO:BP pathways found")

In [ ]:
dc.mt.ulm(data=pdata_filt, net=gobp_pathways_df, layer='normalized')

In [ ]:
pdata_filt.obsm['score_ulm']

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm

# Your model from earlier
model_formula = "pathway_score ~ 1 + ANCESTRY_AFR + ANCESTRY_EAS + ANCESTRY_PEL + d_dx_amm_age + d_dx_amm_bmi + d_pt_sex + CGS_risk"

# Fit for each pathway (column in score_ulm)
results = {}
for pathway in gobp_pathways_df['source'].unique():
    if pathway in pdata_filt.obsm['score_ulm'].columns:
        # Add pathway score to obs for modeling
        pdata_filt.obs['pathway_score'] = pdata_filt.obsm['score_ulm'][pathway]
        
        # Fit model using formula API (R-like syntax)
        model = smf.ols(model_formula, data=pdata_filt.obs).fit()
        
        # Extract coefficient and p-value for ANCESTRY_AFR
        coef = model.params['ANCESTRY_AFR']
        pval = model.pvalues['ANCESTRY_AFR']
        
        results[pathway] = {
            'coefficient': coef,
            'p_value': pval,
            'std_err': model.bse['ANCESTRY_AFR'],
            't_stat': model.tvalues['ANCESTRY_AFR']
        }

In [ ]:
from statsmodels.stats.multitest import multipletests

results_df = pd.DataFrame.from_dict(results, orient='index')
results_df = results_df.sort_values(by='p_value')

# Apply Benjamini-Hochberg FDR correction
results_df['fdr_p_value'] = multipletests(results_df['p_value'], method='fdr_bh')[1]
# Save hallmark results to CSV

results_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_gobp_ulm_regression_results.csv")

print(f"Saved gobp regression results to: {output_dir_fig_tables}{FILE_NAME}_gobp_ulm_regression_results.csv")

print(f"Saved gobp ULM scores to: {output_dir_fig_tables}{FILE_NAME}_gobp_ulm_scores.csv")

# Save gobp ULM scores to CSVgobp_ulm_scores.to_csv(output_dir_fig_tables + f"{FILE_NAME}_gobp_ulm_scores.csv")
gobp_ulm_scores = pd.DataFrame(pdata_filt.obsm['score_ulm'], index=pdata_filt.obs_names)
gobp_ulm_scores.to_csv(output_dir_fig_tables + f"{FILE_NAME}_gobp_ulm_scores.csv")

results_df[results_df['p_value'] < 0.1]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Define color palettes
color_AFR_high = "#86325F"
color_AFR_low = "#5A90C7"
fill_AFR_high = "#9b4472"
fill_AFR_low = "#699ed4"

# Filter for significant pathways
sig_pathways = results_df[results_df['fdr_p_value'] < 0.05].sort_values(by='coefficient').index.tolist()

print(f"Found {len(sig_pathways)} significant pathways (p < 0.05)")

if len(sig_pathways) > 0:
    n_pathways = len(sig_pathways)
    n_cols = 4
    n_rows = (n_pathways + n_cols - 1) // n_cols  # Ceiling division
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes = axes.flatten()
    
    for idx, pathway in enumerate(sig_pathways):
        ax = axes[idx]
        
        # Get ULM scores for this pathway
        pathway_scores = pdata_filt.obsm['score_ulm'][pathway]
        
        # Create a temporary dataframe for plotting
        plot_df = pd.DataFrame({
            'ULM_Score': pathway_scores.values,
            'AFR_BINNED': pdata_filt.obs['AFR_BINNED'].values
        })
        
        # Reorder the categories so AFR High is on the right
        plot_df['AFR_BINNED'] = pd.Categorical(plot_df['AFR_BINNED'], 
                                                categories=['AFR Low', 'AFR High'], 
                                                ordered=True)
        
        # Get the p-value from results_df
        p_val = results_df.loc[pathway, 'p_value']
        
        # Define color palettes for the two groups
        violin_colors = [fill_AFR_low, fill_AFR_high]
        box_colors = [color_AFR_low, color_AFR_high]
        
        # Plot violin plot with transparency
        parts = ax.violinplot([plot_df[plot_df['AFR_BINNED'] == 'AFR Low']['ULM_Score'].values,
                               plot_df[plot_df['AFR_BINNED'] == 'AFR High']['ULM_Score'].values],
                              positions=[0, 1],
                              showmeans=False, 
                              showmedians=False,
                              showextrema=False)
        
        # Color the violin plots with transparency
        for i, pc in enumerate(parts['bodies']):
            pc.set_facecolor(violin_colors[i])
            pc.set_alpha(0.4)
            pc.set_edgecolor('black')
            pc.set_linewidth(1)
        
        # Overlay box plot with custom colors
        bp = ax.boxplot([plot_df[plot_df['AFR_BINNED'] == 'AFR Low']['ULM_Score'].values,
                         plot_df[plot_df['AFR_BINNED'] == 'AFR High']['ULM_Score'].values],
                        positions=[0, 1],
                        widths=0.3,
                        patch_artist=True,
                        showfliers=True)
        
        # Color the box plots
        for i, (patch, color) in enumerate(zip(bp['boxes'], box_colors)):
            patch.set_facecolor(color)
            patch.set_alpha(0.8)
        
        # Color the other box plot elements
        for i, color in enumerate(box_colors):
            bp['medians'][i].set_color('black')
            bp['medians'][i].set_linewidth(2)
            bp['whiskers'][i*2].set_color(color)
            bp['whiskers'][i*2+1].set_color(color)
            bp['caps'][i*2].set_color(color)
            bp['caps'][i*2+1].set_color(color)
            if bp['fliers'][i].get_data()[0].size > 0:
                bp['fliers'][i].set_markerfacecolor(color)
                bp['fliers'][i].set_markeredgecolor(color)
                bp['fliers'][i].set_alpha(0.5)
        
        # Add individual points with matching colors
        for i, category in enumerate(['AFR Low', 'AFR High']):
            y_data = plot_df[plot_df['AFR_BINNED'] == category]['ULM_Score'].values
            x_data = np.random.normal(i, 0.04, size=len(y_data))
            ax.scatter(x_data, y_data, 
                      color=box_colors[i], 
                      alpha=0.3, 
                      s=30,
                      edgecolors='none')
        
        # Calculate position for p-value annotation and expand ylim
        y_max = plot_df['ULM_Score'].max()
        y_min = plot_df['ULM_Score'].min()
        y_range = y_max - y_min
        
        # Position p-value text 5% above the max value
        p_val_y = y_max + 0.05 * y_range
        
        # Expand y-axis to accommodate p-value text (add 12% above max to ensure text fits)
        ax.set_ylim(y_min - 0.05 * y_range, y_max + 0.15 * y_range)
        
        # Format p-value: 3 sig figs if >= 0.001, scientific notation if < 0.001
        if p_val >= 0.001:
            p_val_text = f'p = {p_val:.2g}'
        else:
            p_val_text = f'p = {p_val:.2e}'
        
        ax.text(0.5, p_val_y, 
               p_val_text, 
               ha='center', va='bottom', fontsize=10, fontweight='bold')
        
        # Set labels and title
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['AFR Low', 'AFR High'], fontsize=9)
        ax.set_xlabel('African Genetic Similarity', fontsize=10)
        ax.set_ylabel('ULM Score', fontsize=10)
        ax.set_title(f'{pathway}', fontsize=10, fontweight='bold')
    
    # Hide empty subplots
    for idx in range(n_pathways, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No significant pathways to plot")

<h3> WikiPathways Pathway Analysis </h3>

In [ ]:
#papermill_description=Decoupler_wikipathways

import os
import pickle

# Cache WikiPathways gmt object to disk
wikipathways_cache_file = os.path.join(cache_dir, "wikipathways_c2_cp_2025_1_Hs.pkl")
if os.path.exists(wikipathways_cache_file):
    print(f"Loading WikiPathways from cache: {wikipathways_cache_file}")
    with open(wikipathways_cache_file, 'rb') as f:
        gmt = pickle.load(f)
else:
    print(f"Downloading WikiPathways database...")
    gmt = msig.get_gmt(category="c2.cp.wikipathways", dbver="2025.1.Hs")
    print(f"Saving WikiPathways to cache: {wikipathways_cache_file}")
    with open(wikipathways_cache_file, 'wb') as f:
        pickle.dump(gmt, f)

In [ ]:
# Convert WikiPathways dictionary to pandas DataFrame
# Each row is a gene-pathway pair
pathway_data_wikipathways = []
for pathway_name, genes in gmt.items():
    for gene in genes:
        pathway_data_wikipathways.append({'source': pathway_name, 'target': gene})

wikipathways_pathways_df = pd.DataFrame(pathway_data_wikipathways)
print(f"Created pathway dataframe with {len(wikipathways_pathways_df)} gene-pathway pairs")
print(f"Number of unique pathways: {wikipathways_pathways_df['source'].nunique()}")
print(f"Number of unique genes: {wikipathways_pathways_df['target'].nunique()}")
wikipathways_pathways_df = wikipathways_pathways_df.drop_duplicates(subset=['source','target'])
wikipathways_pathways_df.head()

In [ ]:
# Run
wp_acts, wp_padj = dc.mt.ulm(data=data, net=wikipathways_pathways_df)

if(len(wp_acts.columns) > 0):
    full_combined_df = pd.DataFrame({
        'pathway': wp_acts.columns,
        'score': wp_acts.values.flatten(),
        'padj': wp_padj[wp_acts.columns].values.flatten()
    })
else:
    full_combined_df = pd.DataFrame(columns=['pathway','score','padj'])

full_combined_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_wp_pathways_score_unfiltered.tsv", sep='\t')



# Filter by sign padj
msk = (wp_padj.T < 0.05).iloc[:, 0]
wp_acts = wp_acts.loc[:, msk]

wp_acts

In [ ]:
# Combine wp_acts and wp_padj
if(len(wp_acts.columns) > 0):
    combined_df = pd.DataFrame({
        'pathway': wp_acts.columns,
        'score': wp_acts.values.flatten(),
        'padj': wp_padj[wp_acts.columns].values.flatten()
    })
else:
    combined_df = pd.DataFrame(columns=['pathway','score','padj'])

In [ ]:
if(len(wp_acts.columns) > 0):
    dc.pl.barplot(data=wp_acts, name="AFRHigh_vs_AFRLow", figsize=(10, min(len(wp_acts.columns) * 3/5, 50*3/5)), top=50)
else:
    print("No significant WikiPathways pathways found")

In [ ]:
# Add leading edge genes to combined_df
if len(wp_acts.columns) > 0:
    leading_edge_dict = {}
    for pathway in wp_acts.columns.tolist():
        _, le = dc.pl.leading_edge(
            merged_df,
            stat="stat",
            net=wikipathways_pathways_df,
            name=pathway,
        )
        leading_edge_dict[pathway] = ', '.join(le)
    
    # Add as a new column
    combined_df['leading_edge_genes'] = combined_df['pathway'].map(leading_edge_dict)
else:
    print("No significant WikiPathways pathways found")

In [ ]:
combined_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_wikipathways_pathways_score_fdr_legenes.tsv", sep='\t')
combined_df

In [ ]:
if(len(wp_acts.columns) > 0):
    
    n_pathways = len(wp_acts.columns)
    if n_pathways > 1:
        n_cols = 4
        n_rows = (n_pathways + n_cols - 1) // n_cols  # Ceiling division
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
        axes = axes.flatten()
        
        for idx, pathway in enumerate(wp_acts.columns.tolist()):
            dc.pl.volcano(
                data=merged_df,
                x="log2FoldChange",
                y="padj",
                net=wikipathways_pathways_df,
                name=pathway,
                top=30,
                thr_stat=0.1,
                thr_sign=0.1,
                ax=axes[idx]
            )
            axes[idx].set_title(pathway, fontsize=10)
        
        # Hide empty subplots
        for idx in range(n_pathways, len(axes)):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()
    elif(n_pathways==1):
        for pathway in wp_acts.columns.tolist():
            dc.pl.volcano(
                data=merged_df,
                x="log2FoldChange",
                y="padj",
                net=wikipathways_pathways_df,
                name=pathway,
                top=30,
                thr_stat=0.1,
                thr_sign=0.1
            )


else:
    print("No significant WikiPathways pathways found")

In [ ]:
dc.mt.ulm(data=pdata_filt, net=wikipathways_pathways_df, layer='normalized')

In [ ]:
pdata_filt.obsm['score_ulm']

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm

# Your model from earlier
model_formula = "pathway_score ~ 1 + ANCESTRY_AFR + ANCESTRY_EAS + ANCESTRY_PEL + d_dx_amm_age + d_dx_amm_bmi + d_pt_sex + CGS_risk"

# Fit for each pathway (column in score_ulm)
results = {}
for pathway in wikipathways_pathways_df['source'].unique():
    if pathway in pdata_filt.obsm['score_ulm'].columns:
        # Add pathway score to obs for modeling
        pdata_filt.obs['pathway_score'] = pdata_filt.obsm['score_ulm'][pathway]
        
        # Fit model using formula API (R-like syntax)
        model = smf.ols(model_formula, data=pdata_filt.obs).fit()
        
        # Extract coefficient and p-value for ANCESTRY_AFR
        coef = model.params['ANCESTRY_AFR']
        pval = model.pvalues['ANCESTRY_AFR']
        
        results[pathway] = {
            'coefficient': coef,
            'p_value': pval,
            'std_err': model.bse['ANCESTRY_AFR'],
            't_stat': model.tvalues['ANCESTRY_AFR']
        }

In [ ]:
from statsmodels.stats.multitest import multipletests

results_df = pd.DataFrame.from_dict(results, orient='index')
results_df = results_df.sort_values(by='p_value')

# Apply Benjamini-Hochberg FDR correction
results_df['fdr_p_value'] = multipletests(results_df['p_value'], method='fdr_bh')[1]

results_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_wikipathways_ulm_regression_results.csv")

print(f"Saved wikipathways regression results to: {output_dir_fig_tables}{FILE_NAME}_wikipathways_ulm_regression_results.csv")

print(f"Saved wikipathways ULM scores to: {output_dir_fig_tables}{FILE_NAME}_wikipathways_ulm_scores.csv")

wikipathways_ulm_scores = pd.DataFrame(pdata_filt.obsm['score_ulm'], index=pdata_filt.obs_names)
wikipathways_ulm_scores.to_csv(output_dir_fig_tables + f"{FILE_NAME}_wikipathways_ulm_scores.csv")

results_df[results_df['p_value'] < 0.1]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Define color palettes
color_AFR_high = "#86325F"
color_AFR_low = "#5A90C7"
fill_AFR_high = "#9b4472"
fill_AFR_low = "#699ed4"

# Filter for significant pathways
sig_pathways = results_df[results_df['fdr_p_value'] < 0.05].sort_values(by='coefficient').index.tolist()

print(f"Found {len(sig_pathways)} significant pathways (p < 0.05)")

if len(sig_pathways) > 0:
    n_pathways = len(sig_pathways)
    n_cols = 4
    n_rows = (n_pathways + n_cols - 1) // n_cols  # Ceiling division
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes = axes.flatten()
    
    for idx, pathway in enumerate(sig_pathways):
        ax = axes[idx]
        
        # Get ULM scores for this pathway
        pathway_scores = pdata_filt.obsm['score_ulm'][pathway]
        
        # Create a temporary dataframe for plotting
        plot_df = pd.DataFrame({
            'ULM_Score': pathway_scores.values,
            'AFR_BINNED': pdata_filt.obs['AFR_BINNED'].values
        })
        
        # Reorder the categories so AFR High is on the right
        plot_df['AFR_BINNED'] = pd.Categorical(plot_df['AFR_BINNED'], 
                                                categories=['AFR Low', 'AFR High'], 
                                                ordered=True)
        
        # Get the p-value from results_df
        p_val = results_df.loc[pathway, 'p_value']
        
        # Define color palettes for the two groups
        violin_colors = [fill_AFR_low, fill_AFR_high]
        box_colors = [color_AFR_low, color_AFR_high]
        
        # Plot violin plot with transparency
        parts = ax.violinplot([plot_df[plot_df['AFR_BINNED'] == 'AFR Low']['ULM_Score'].values,
                               plot_df[plot_df['AFR_BINNED'] == 'AFR High']['ULM_Score'].values],
                              positions=[0, 1],
                              showmeans=False, 
                              showmedians=False,
                              showextrema=False)
        
        # Color the violin plots with transparency
        for i, pc in enumerate(parts['bodies']):
            pc.set_facecolor(violin_colors[i])
            pc.set_alpha(0.4)
            pc.set_edgecolor('black')
            pc.set_linewidth(1)
        
        # Overlay box plot with custom colors
        bp = ax.boxplot([plot_df[plot_df['AFR_BINNED'] == 'AFR Low']['ULM_Score'].values,
                         plot_df[plot_df['AFR_BINNED'] == 'AFR High']['ULM_Score'].values],
                        positions=[0, 1],
                        widths=0.3,
                        patch_artist=True,
                        showfliers=True)
        
        # Color the box plots
        for i, (patch, color) in enumerate(zip(bp['boxes'], box_colors)):
            patch.set_facecolor(color)
            patch.set_alpha(0.8)
        
        # Color the other box plot elements
        for i, color in enumerate(box_colors):
            bp['medians'][i].set_color('black')
            bp['medians'][i].set_linewidth(2)
            bp['whiskers'][i*2].set_color(color)
            bp['whiskers'][i*2+1].set_color(color)
            bp['caps'][i*2].set_color(color)
            bp['caps'][i*2+1].set_color(color)
            if bp['fliers'][i].get_data()[0].size > 0:
                bp['fliers'][i].set_markerfacecolor(color)
                bp['fliers'][i].set_markeredgecolor(color)
                bp['fliers'][i].set_alpha(0.5)
        
        # Add individual points with matching colors
        for i, category in enumerate(['AFR Low', 'AFR High']):
            y_data = plot_df[plot_df['AFR_BINNED'] == category]['ULM_Score'].values
            x_data = np.random.normal(i, 0.04, size=len(y_data))
            ax.scatter(x_data, y_data, 
                      color=box_colors[i], 
                      alpha=0.3, 
                      s=30,
                      edgecolors='none')
        
        # Calculate position for p-value annotation and expand ylim
        y_max = plot_df['ULM_Score'].max()
        y_min = plot_df['ULM_Score'].min()
        y_range = y_max - y_min
        
        # Position p-value text 5% above the max value
        p_val_y = y_max + 0.05 * y_range
        
        # Expand y-axis to accommodate p-value text (add 12% above max to ensure text fits)
        ax.set_ylim(y_min - 0.05 * y_range, y_max + 0.15 * y_range)
        
        # Format p-value: 3 sig figs if >= 0.001, scientific notation if < 0.001
        if p_val >= 0.001:
            p_val_text = f'p = {p_val:.2g}'
        else:
            p_val_text = f'p = {p_val:.2e}'
        
        ax.text(0.5, p_val_y, 
               p_val_text, 
               ha='center', va='bottom', fontsize=10, fontweight='bold')
        
        # Set labels and title
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['AFR Low', 'AFR High'], fontsize=9)
        ax.set_xlabel('African Genetic Similarity', fontsize=10)
        ax.set_ylabel('ULM Score', fontsize=10)
        ax.set_title(f'{pathway}', fontsize=10, fontweight='bold')
    
    # Hide empty subplots
    for idx in range(n_pathways, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No significant pathways to plot")

<h3> KEGG Pathway Analysis </h3>

In [ ]:
#papermill_description=Decoupler_KEGG
import os
import pickle

# Cache KEGG gmt object to disk
kegg_cache_file = os.path.join(cache_dir, "kegg_medicus_2025_1_Hs.pkl")
if os.path.exists(kegg_cache_file):
    print(f"Loading KEGG from cache: {kegg_cache_file}")
    with open(kegg_cache_file, 'rb') as f:
        gmt = pickle.load(f)
else:
    print(f"Downloading KEGG database...")
    gmt = msig.get_gmt(category="c2.cp.kegg_medicus", dbver="2025.1.Hs")
    print(f"Saving KEGG to cache: {kegg_cache_file}")
    with open(kegg_cache_file, 'wb') as f:
        pickle.dump(gmt, f)

In [ ]:
# Convert KEGG dictionary to pandas DataFrame
# Each row is a gene-pathway pair
pathway_data_kegg = []
for pathway_name, genes in gmt.items():
    for gene in genes:
        pathway_data_kegg.append({'source': pathway_name, 'target': gene})

kegg_pathways_df = pd.DataFrame(pathway_data_kegg)
print(f"Created pathway dataframe with {len(kegg_pathways_df)} gene-pathway pairs")
print(f"Number of unique pathways: {kegg_pathways_df['source'].nunique()}")
print(f"Number of unique genes: {kegg_pathways_df['target'].nunique()}")
kegg_pathways_df = kegg_pathways_df.drop_duplicates(subset=['source','target'])
kegg_pathways_df.head()

In [ ]:
# Run
kegg_acts, kegg_padj = dc.mt.ulm(data=data, net=kegg_pathways_df)

if(len(kegg_acts.columns) > 0):
    full_combined_df = pd.DataFrame({
        'pathway': kegg_acts.columns,
        'score': kegg_acts.values.flatten(),
        'padj': kegg_padj[kegg_acts.columns].values.flatten()
    })
else:
    full_combined_df = pd.DataFrame(columns=['pathway','score','padj'])

full_combined_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_kegg_pathways_score_unfiltered.tsv", sep='\t')



# Filter by sign padj
msk = (kegg_padj.T < 0.05).iloc[:, 0]
kegg_acts = kegg_acts.loc[:, msk]

kegg_acts

In [ ]:
# Combine kegg_acts and kegg_padj
if(len(kegg_acts.columns) > 0):
    combined_df = pd.DataFrame({
        'pathway': kegg_acts.columns,
        'score': kegg_acts.values.flatten(),
        'padj': kegg_padj[kegg_acts.columns].values.flatten()
    })
else:
    combined_df = pd.DataFrame(columns=['pathway','score','padj'])

In [ ]:
if(len(kegg_acts.columns) > 0):
    dc.pl.barplot(data=kegg_acts, name="AFRHigh_vs_AFRLow", figsize=(10, min(len(kegg_acts.columns) * 3/5, 50*3/5)), top=50)
else:
    print("No significant KEGG pathways found")

In [ ]:
# Add leading edge genes to combined_df
if len(kegg_acts.columns) > 0:
    leading_edge_dict = {}
    
    n_pathways = len(kegg_acts.columns)

    for pathway in kegg_acts.columns.tolist():
        _, le = dc.pl.leading_edge(
            merged_df,
            stat="stat",
            net=kegg_pathways_df,
            name=pathway,
        )
        leading_edge_dict[pathway] = ', '.join(le)

    # Add as a new column
    combined_df['leading_edge_genes'] = combined_df['pathway'].map(leading_edge_dict)
else:
    print("No significant KEGG pathways found")


In [ ]:
combined_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_kegg_pathways_score_fdr_legenes.tsv", sep='\t')
combined_df

In [ ]:
if(len(kegg_acts.columns) > 0):
    
    n_pathways = len(kegg_acts.columns)
    if n_pathways > 1:
        n_cols = 4
        n_rows = (n_pathways + n_cols - 1) // n_cols  # Ceiling division
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
        axes = axes.flatten() 
        
        for idx, pathway in enumerate(kegg_acts.columns.tolist()):
            dc.pl.volcano(
                data=merged_df,
                x="log2FoldChange",
                y="padj",
                net=kegg_pathways_df,
                name=pathway,
                top=30,
                thr_stat=0.1,
                thr_sign=0.1,
                ax=axes[idx]
            )
            axes[idx].set_title(pathway, fontsize=10)
        
        # Hide empty subplots
        for idx in range(n_pathways, len(axes)):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()
    elif(n_pathways==1):
        for pathway in kegg_acts.columns.tolist():
            dc.pl.volcano(
                data=merged_df,
                x="log2FoldChange",
                y="padj",
                net=kegg_pathways_df,
                name=pathway,
                top=30,
                thr_stat=0.1,
                thr_sign=0.1
            )


else:
    print("No significant KEGG pathways found")

In [ ]:
dc.mt.ulm(data=pdata_filt, net=kegg_pathways_df, layer='normalized')

In [ ]:
pdata_filt.obsm['score_ulm']

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm

# Your model from earlier
model_formula = "pathway_score ~ 1 + ANCESTRY_AFR + ANCESTRY_EAS + ANCESTRY_PEL + d_dx_amm_age + d_dx_amm_bmi + d_pt_sex + CGS_risk"

# Fit for each pathway (column in score_ulm)
results = {}
for pathway in kegg_pathways_df['source'].unique():
    if pathway in pdata_filt.obsm['score_ulm'].columns:
        # Add pathway score to obs for modeling
        pdata_filt.obs['pathway_score'] = pdata_filt.obsm['score_ulm'][pathway]
        
        # Fit model using formula API (R-like syntax)
        model = smf.ols(model_formula, data=pdata_filt.obs).fit()
        
        # Extract coefficient and p-value for ANCESTRY_AFR
        coef = model.params['ANCESTRY_AFR']
        pval = model.pvalues['ANCESTRY_AFR']
        
        results[pathway] = {
            'coefficient': coef,
            'p_value': pval,
            'std_err': model.bse['ANCESTRY_AFR'],
            't_stat': model.tvalues['ANCESTRY_AFR']
        }

In [ ]:
from statsmodels.stats.multitest import multipletests

results_df = pd.DataFrame.from_dict(results, orient='index')
results_df = results_df.sort_values(by='p_value')

# Apply Benjamini-Hochberg FDR correction
results_df['fdr_p_value'] = multipletests(results_df['p_value'], method='fdr_bh')[1]

results_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_kegg_ulm_regression_results.csv")

print(f"Saved kegg regression results to: {output_dir_fig_tables}{FILE_NAME}_kegg_ulm_regression_results.csv")

print(f"Saved kegg ULM scores to: {output_dir_fig_tables}{FILE_NAME}_kegg_ulm_scores.csv")

kegg_ulm_scores = pd.DataFrame(pdata_filt.obsm['score_ulm'], index=pdata_filt.obs_names)
kegg_ulm_scores.to_csv(output_dir_fig_tables + f"{FILE_NAME}_kegg_ulm_scores.csv")

results_df[results_df['p_value'] < 0.1]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Define color palettes
color_AFR_high = "#86325F"
color_AFR_low = "#5A90C7"
fill_AFR_high = "#9b4472"
fill_AFR_low = "#699ed4"

# Filter for significant pathways
sig_pathways = results_df[results_df['fdr_p_value'] < 0.05].sort_values(by='coefficient').index.tolist()

print(f"Found {len(sig_pathways)} significant pathways (p < 0.05)")

if len(sig_pathways) > 0:
    n_pathways = len(sig_pathways)
    n_cols = 4
    n_rows = (n_pathways + n_cols - 1) // n_cols  # Ceiling division
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes = axes.flatten() 
    
    for idx, pathway in enumerate(sig_pathways):
        ax = axes[idx]
        
        # Get ULM scores for this pathway
        pathway_scores = pdata_filt.obsm['score_ulm'][pathway]
        
        # Create a temporary dataframe for plotting
        plot_df = pd.DataFrame({
            'ULM_Score': pathway_scores.values,
            'AFR_BINNED': pdata_filt.obs['AFR_BINNED'].values
        })
        
        # Reorder the categories so AFR High is on the right
        plot_df['AFR_BINNED'] = pd.Categorical(plot_df['AFR_BINNED'], 
                                                categories=['AFR Low', 'AFR High'], 
                                                ordered=True)
        
        # Get the p-value from results_df
        p_val = results_df.loc[pathway, 'p_value']
        
        # Define color palettes for the two groups
        violin_colors = [fill_AFR_low, fill_AFR_high]
        box_colors = [color_AFR_low, color_AFR_high]
        
        # Plot violin plot with transparency
        parts = ax.violinplot([plot_df[plot_df['AFR_BINNED'] == 'AFR Low']['ULM_Score'].values,
                               plot_df[plot_df['AFR_BINNED'] == 'AFR High']['ULM_Score'].values],
                              positions=[0, 1],
                              showmeans=False, 
                              showmedians=False,
                              showextrema=False)
        
        # Color the violin plots with transparency
        for i, pc in enumerate(parts['bodies']):
            pc.set_facecolor(violin_colors[i])
            pc.set_alpha(0.4)
            pc.set_edgecolor('black')
            pc.set_linewidth(1)
        
        # Overlay box plot with custom colors
        bp = ax.boxplot([plot_df[plot_df['AFR_BINNED'] == 'AFR Low']['ULM_Score'].values,
                         plot_df[plot_df['AFR_BINNED'] == 'AFR High']['ULM_Score'].values],
                        positions=[0, 1],
                        widths=0.3,
                        patch_artist=True,
                        showfliers=True)
        
        # Color the box plots
        for i, (patch, color) in enumerate(zip(bp['boxes'], box_colors)):
            patch.set_facecolor(color)
            patch.set_alpha(0.8)
        
        # Color the other box plot elements
        for i, color in enumerate(box_colors):
            bp['medians'][i].set_color('black')
            bp['medians'][i].set_linewidth(2)
            bp['whiskers'][i*2].set_color(color)
            bp['whiskers'][i*2+1].set_color(color)
            bp['caps'][i*2].set_color(color)
            bp['caps'][i*2+1].set_color(color)
            if bp['fliers'][i].get_data()[0].size > 0:
                bp['fliers'][i].set_markerfacecolor(color)
                bp['fliers'][i].set_markeredgecolor(color)
                bp['fliers'][i].set_alpha(0.5)
        
        # Add individual points with matching colors
        for i, category in enumerate(['AFR Low', 'AFR High']):
            y_data = plot_df[plot_df['AFR_BINNED'] == category]['ULM_Score'].values
            x_data = np.random.normal(i, 0.04, size=len(y_data))
            ax.scatter(x_data, y_data, 
                      color=box_colors[i], 
                      alpha=0.3, 
                      s=30,
                      edgecolors='none')
        
        # Calculate position for p-value annotation and expand ylim
        y_max = plot_df['ULM_Score'].max()
        y_min = plot_df['ULM_Score'].min()
        y_range = y_max - y_min
        
        # Position p-value text 5% above the max value
        p_val_y = y_max + 0.05 * y_range
        
        # Expand y-axis to accommodate p-value text (add 12% above max to ensure text fits)
        ax.set_ylim(y_min - 0.05 * y_range, y_max + 0.15 * y_range)
        
        # Format p-value: 3 sig figs if >= 0.001, scientific notation if < 0.001
        if p_val >= 0.001:
            p_val_text = f'p = {p_val:.2g}'
        else:
            p_val_text = f'p = {p_val:.2e}'
        
        ax.text(0.5, p_val_y, 
               p_val_text, 
               ha='center', va='bottom', fontsize=10, fontweight='bold')
        
        # Set labels and title
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['AFR Low', 'AFR High'], fontsize=9)
        ax.set_xlabel('African Genetic Similarity', fontsize=10)
        ax.set_ylabel('ULM Score', fontsize=10)
        ax.set_title(f'{pathway}', fontsize=10, fontweight='bold')
    
    # Hide empty subplots
    for idx in range(n_pathways, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No significant pathways to plot")

<h3> Custom Gene Sets </h3>

In [ ]:
#papermill_description=Decoupler_Custom

# Convert CUSTOM_PATHWAYS dictionary to pandas DataFrame
# Each row is a gene-pathway pair
pathway_data = []
for pathway_name, genes in CUSTOM_PATHWAYS.items():
    for gene in genes:
        pathway_data.append({'source': pathway_name, 'target': gene})

custom_pathways_df = pd.DataFrame(pathway_data)
print(f"Created pathway dataframe with {len(custom_pathways_df)} gene-pathway pairs")
print(f"Number of unique pathways: {custom_pathways_df['source'].nunique()}")
print(f"Number of unique genes: {custom_pathways_df['target'].nunique()}")
custom_pathways_df = custom_pathways_df.drop_duplicates(subset=['source','target'])
custom_pathways_df.head()

Assessing ulm's behavior on these custom sets. Plan to compare ULM vs GSVA long term if needed regardless of results here

In [ ]:
# Run
custom_acts, custom_padj = dc.mt.ulm(data=data, net=custom_pathways_df)

if(len(custom_acts.columns) > 0):
    full_combined_df = pd.DataFrame({
        'pathway': custom_acts.columns,
        'score': custom_acts.values.flatten(),
        'padj': custom_padj[custom_acts.columns].values.flatten()
    })
else:
    full_combined_df = pd.DataFrame(columns=['pathway','score','padj'])

full_combined_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_custom_pathways_score_unfiltered.tsv", sep='\t')


# Filter by sign padj
msk = (custom_padj.T < 0.05).iloc[:, 0]
custom_acts = custom_acts.loc[:, msk]
custom_acts

In [ ]:
if(len(custom_acts.columns) > 0):
    dc.pl.barplot(data=custom_acts, name="AFRHigh_vs_AFRLow", figsize=(6, 6), top=50)
else:
    print("No significant CUSTOM pathways found")

In [ ]:
if(len(custom_acts.columns) > 0):
    
    n_pathways = len(custom_acts.columns)

    for pathway in custom_acts.columns.tolist():
        _, le = dc.pl.leading_edge(
            merged_df,
            stat="stat",
            net=custom_pathways_df,
            name=pathway,
        )
        print(f"{pathway} leading edge:", le[:10])


else:
    print("No significant custom pathways found")


In [ ]:
if(len(custom_acts.columns) > 0):
    for pathway in custom_acts.columns.tolist():
        dc.pl.volcano(
            data=merged_df,
            x="log2FoldChange",
            y="padj",
            net=custom_pathways_df,
            name=pathway,
            top=20,
            thr_stat = 0.25,
            thr_sign = 0.1
        )
else:
    print("No significant custom pathways found")

ULM Based Approach

In [ ]:
dc.mt.ulm(data=pdata_filt, net=custom_pathways_df, layer='normalized')

In [ ]:
pdata_filt.obsm['score_ulm']

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm

# Your model from earlier
model_formula = "pathway_score ~ 1 + ANCESTRY_AFR + ANCESTRY_EAS + ANCESTRY_PEL + d_dx_amm_age + d_dx_amm_bmi + d_pt_sex + CGS_risk"

# Fit for each pathway (column in score_ulm)
results = {}
for pathway in custom_pathways_df['source'].unique():
    if pathway in pdata_filt.obsm['score_ulm'].columns:
        # Add pathway score to obs for modeling
        pdata_filt.obs['pathway_score'] = pdata_filt.obsm['score_ulm'][pathway]
        
        # Fit model using formula API (R-like syntax)
        model = smf.ols(model_formula, data=pdata_filt.obs).fit()
        
        # Extract coefficient and p-value for ANCESTRY_AFR
        coef = model.params['ANCESTRY_AFR']
        pval = model.pvalues['ANCESTRY_AFR']
        
        results[pathway] = {
            'coefficient': coef,
            'p_value': pval,
            'std_err': model.bse['ANCESTRY_AFR'],
            't_stat': model.tvalues['ANCESTRY_AFR']
        }

In [ ]:
from statsmodels.stats.multitest import multipletests

results_df = pd.DataFrame.from_dict(results, orient='index')
results_df = results_df.sort_values(by='p_value')

# Apply Benjamini-Hochberg FDR correction
results_df['fdr_p_value'] = multipletests(results_df['p_value'], method='fdr_bh')[1]

# Save custom pathways results to CSV

results_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_custom_pathways_ulm_regression_results.csv")

print(f"Saved custom pathways regression results to: {output_dir_fig_tables}{FILE_NAME}_custom_pathways_ulm_regression_results.csv")

print(f"Saved custom pathways ULM scores to: {output_dir_fig_tables}{FILE_NAME}_custom_pathways_ulm_scores.csv")

custom_ulm_scores = pd.DataFrame(pdata_filt.obsm['score_ulm'], index=pdata_filt.obs_names)
custom_ulm_scores.to_csv(output_dir_fig_tables + f"{FILE_NAME}_custom_pathways_ulm_scores.csv")

results_df[results_df['p_value'] < 0.1]

Box/Violin of custom pathways with significant results

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Define color palettes
color_AFR_high = "#86325F"
color_AFR_low = "#5A90C7"
fill_AFR_high = "#9b4472"
fill_AFR_low = "#699ed4"

# Filter for significant pathways
sig_pathways = results_df[results_df['p_value'] < 0.05].sort_values(by='coefficient').index.tolist()
other_pathways = results_df[results_df['p_value'] >= 0.05].sort_values(by='coefficient').index.tolist()

print(f"Found {len(sig_pathways)} significant pathways (raw p < 0.05)")

if len(sig_pathways) > 0:
    n_pathways = len(sig_pathways)
    n_cols = 4
    n_rows = (n_pathways + n_cols - 1) // n_cols  # Ceiling division
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes = axes.flatten()
    
    for idx, pathway in enumerate(sig_pathways):
        ax = axes[idx]
        
        # Get ULM scores for this pathway
        pathway_scores = pdata_filt.obsm['score_ulm'][pathway]
        
        # Create a temporary dataframe for plotting
        plot_df = pd.DataFrame({
            'ULM_Score': pathway_scores.values,
            'AFR_BINNED': pdata_filt.obs['AFR_BINNED'].values
        })
        
        # Reorder the categories so AFR High is on the right
        plot_df['AFR_BINNED'] = pd.Categorical(plot_df['AFR_BINNED'], 
                                                categories=['AFR Low', 'AFR High'], 
                                                ordered=True)
        
        # Get the p-value from results_df
        p_val = results_df.loc[pathway, 'p_value']
        
        # Define color palettes for the two groups
        violin_colors = [fill_AFR_low, fill_AFR_high]
        box_colors = [color_AFR_low, color_AFR_high]
        
        # Plot violin plot with transparency
        parts = ax.violinplot([plot_df[plot_df['AFR_BINNED'] == 'AFR Low']['ULM_Score'].values,
                               plot_df[plot_df['AFR_BINNED'] == 'AFR High']['ULM_Score'].values],
                              positions=[0, 1],
                              showmeans=False, 
                              showmedians=False,
                              showextrema=False)
        
        # Color the violin plots with transparency
        for i, pc in enumerate(parts['bodies']):
            pc.set_facecolor(violin_colors[i])
            pc.set_alpha(0.4)
            pc.set_edgecolor('black')
            pc.set_linewidth(1)
        
        # Overlay box plot with custom colors
        bp = ax.boxplot([plot_df[plot_df['AFR_BINNED'] == 'AFR Low']['ULM_Score'].values,
                         plot_df[plot_df['AFR_BINNED'] == 'AFR High']['ULM_Score'].values],
                        positions=[0, 1],
                        widths=0.3,
                        patch_artist=True,
                        showfliers=True)
        
        # Color the box plots
        for i, (patch, color) in enumerate(zip(bp['boxes'], box_colors)):
            patch.set_facecolor(color)
            patch.set_alpha(0.8)
        
        # Color the other box plot elements
        for i, color in enumerate(box_colors):
            bp['medians'][i].set_color('black')
            bp['medians'][i].set_linewidth(2)
            bp['whiskers'][i*2].set_color(color)
            bp['whiskers'][i*2+1].set_color(color)
            bp['caps'][i*2].set_color(color)
            bp['caps'][i*2+1].set_color(color)
            if bp['fliers'][i].get_data()[0].size > 0:
                bp['fliers'][i].set_markerfacecolor(color)
                bp['fliers'][i].set_markeredgecolor(color)
                bp['fliers'][i].set_alpha(0.5)
        
        # Add individual points with matching colors
        for i, category in enumerate(['AFR Low', 'AFR High']):
            y_data = plot_df[plot_df['AFR_BINNED'] == category]['ULM_Score'].values
            x_data = np.random.normal(i, 0.04, size=len(y_data))
            ax.scatter(x_data, y_data, 
                      color=box_colors[i], 
                      alpha=0.3, 
                      s=30,
                      edgecolors='none')
        
        # Calculate position for p-value annotation and expand ylim
        y_max = plot_df['ULM_Score'].max()
        y_min = plot_df['ULM_Score'].min()
        y_range = y_max - y_min
        
        # Position p-value text 5% above the max value
        p_val_y = y_max + 0.05 * y_range
        
        # Expand y-axis to accommodate p-value text (add 12% above max to ensure text fits)
        ax.set_ylim(y_min - 0.05 * y_range, y_max + 0.15 * y_range)
        
        # Format p-value: 3 sig figs if >= 0.001, scientific notation if < 0.001
        if p_val >= 0.001:
            p_val_text = f'p = {p_val:.2g}'
        else:
            p_val_text = f'p = {p_val:.2e}'
        
        ax.text(0.5, p_val_y, 
               p_val_text, 
               ha='center', va='bottom', fontsize=10, fontweight='bold')
        
        # Set labels and title
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['AFR Low', 'AFR High'], fontsize=9)
        ax.set_xlabel('African Genetic Similarity', fontsize=10)
        ax.set_ylabel('ULM Score', fontsize=10)
        ax.set_title(f'{pathway}', fontsize=10, fontweight='bold')
    
    # Hide empty subplots
    for idx in range(n_pathways, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No significant pathways to plot")

Box/Violin of all other custom pathways

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Define color palettes
color_AFR_high = "#86325F"
color_AFR_low = "#5A90C7"
fill_AFR_high = "#9b4472"
fill_AFR_low = "#699ed4"

# Filter for non-significant pathways
other_pathways = results_df[results_df['p_value'] >= 0.05].sort_values(by='coefficient').index.tolist()

print(f"Found {len(other_pathways)} non-significant pathways (raw p >= 0.05)")

if len(other_pathways) > 0:
    n_pathways = len(other_pathways)
    n_cols = 4
    n_rows = (n_pathways + n_cols - 1) // n_cols  # Ceiling division
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes = axes.flatten()
    
    for idx, pathway in enumerate(other_pathways):
        ax = axes[idx]
        
        # Get ULM scores for this pathway
        pathway_scores = pdata_filt.obsm['score_ulm'][pathway]
        
        # Create a temporary dataframe for plotting
        plot_df = pd.DataFrame({
            'ULM_Score': pathway_scores.values,
            'AFR_BINNED': pdata_filt.obs['AFR_BINNED'].values
        })
        
        # Reorder the categories so AFR High is on the right
        plot_df['AFR_BINNED'] = pd.Categorical(plot_df['AFR_BINNED'], 
                                                categories=['AFR Low', 'AFR High'], 
                                                ordered=True)
        
        # Get the p-value from results_df
        p_val = results_df.loc[pathway, 'p_value']
        
        # Define color palettes for the two groups
        violin_colors = [fill_AFR_low, fill_AFR_high]
        box_colors = [color_AFR_low, color_AFR_high]
        
        # Plot violin plot with transparency
        parts = ax.violinplot([plot_df[plot_df['AFR_BINNED'] == 'AFR Low']['ULM_Score'].values,
                               plot_df[plot_df['AFR_BINNED'] == 'AFR High']['ULM_Score'].values],
                              positions=[0, 1],
                              showmeans=False, 
                              showmedians=False,
                              showextrema=False)
        
        # Color the violin plots with transparency
        for i, pc in enumerate(parts['bodies']):
            pc.set_facecolor(violin_colors[i])
            pc.set_alpha(0.4)
            pc.set_edgecolor('black')
            pc.set_linewidth(1)
        
        # Overlay box plot with custom colors
        bp = ax.boxplot([plot_df[plot_df['AFR_BINNED'] == 'AFR Low']['ULM_Score'].values,
                         plot_df[plot_df['AFR_BINNED'] == 'AFR High']['ULM_Score'].values],
                        positions=[0, 1],
                        widths=0.3,
                        patch_artist=True,
                        showfliers=True)
        
        # Color the box plots
        for i, (patch, color) in enumerate(zip(bp['boxes'], box_colors)):
            patch.set_facecolor(color)
            patch.set_alpha(0.8)
        
        # Color the other box plot elements
        for i, color in enumerate(box_colors):
            bp['medians'][i].set_color('black')
            bp['medians'][i].set_linewidth(2)
            bp['whiskers'][i*2].set_color(color)
            bp['whiskers'][i*2+1].set_color(color)
            bp['caps'][i*2].set_color(color)
            bp['caps'][i*2+1].set_color(color)
            if bp['fliers'][i].get_data()[0].size > 0:
                bp['fliers'][i].set_markerfacecolor(color)
                bp['fliers'][i].set_markeredgecolor(color)
                bp['fliers'][i].set_alpha(0.5)
        
        # Add individual points with matching colors
        for i, category in enumerate(['AFR Low', 'AFR High']):
            y_data = plot_df[plot_df['AFR_BINNED'] == category]['ULM_Score'].values
            x_data = np.random.normal(i, 0.04, size=len(y_data))
            ax.scatter(x_data, y_data, 
                      color=box_colors[i], 
                      alpha=0.3, 
                      s=30,
                      edgecolors='none')
        
        # Calculate position for p-value annotation and expand ylim
        y_max = plot_df['ULM_Score'].max()
        y_min = plot_df['ULM_Score'].min()
        y_range = y_max - y_min
        
        # Position p-value text 5% above the max value
        p_val_y = y_max + 0.05 * y_range
        
        # Expand y-axis to accommodate p-value text (add 12% above max to ensure text fits)
        ax.set_ylim(y_min - 0.05 * y_range, y_max + 0.15 * y_range)
        
        # Format p-value: 3 sig figs if >= 0.001, scientific notation if < 0.001
        if p_val >= 0.001:
            p_val_text = f'p = {p_val:.2g}'
        else:
            p_val_text = f'p = {p_val:.2e}'
        
        ax.text(0.5, p_val_y, 
               p_val_text, 
               ha='center', va='bottom', fontsize=10, fontweight='bold')
        
        # Set labels and title
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['AFR Low', 'AFR High'], fontsize=9)
        ax.set_xlabel('African Genetic Similarity', fontsize=10)
        ax.set_ylabel('ULM Score', fontsize=10)
        ax.set_title(f'{pathway}', fontsize=10, fontweight='bold')
    
    # Hide empty subplots
    for idx in range(n_pathways, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No non-significant pathways to plot")

<h3> CytoSig </h3>

In [ ]:
#papermill_description=Decoupler_CytoSig

import urllib
from pathlib import Path

cytosig_file = Path(cache_dir + "/cytosig_signature.csv")

# Retrieve CytoSig signature
if cytosig_file.exists():
    cytosig_signature = pd.read_csv(cytosig_file, sep="\t")
else:
    urllib.request.urlretrieve(
        "https://github.com/data2intelligence/CytoSig/raw/master/CytoSig/signature.centroid.expand",
        cytosig_file,
    )
    cytosig_signature = pd.read_csv(cytosig_file, sep="\t")


cyto_sig = pd.melt(
    cytosig_signature.rename_axis("target").reset_index(),
    var_name="source",
    id_vars=["target"],
    value_name="weight",
).reindex(columns=["source", "target", "weight"])
cyto_sig

In [ ]:
# Run
cs_acts, cs_padj = dc.mt.ulm(data=data, net=cyto_sig)
if(len(cs_acts.columns) > 0):
    full_combined_df = pd.DataFrame({
        'pathway': cs_acts.columns,
        'score': cs_acts.values.flatten(),
        'padj': cs_padj[cs_acts.columns].values.flatten()
    })
else:
    full_combined_df = pd.DataFrame(columns=['pathway','score','padj'])

full_combined_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_cytosig_pathways_score_unfiltered.tsv", sep='\t')


# Filter by sign padj
msk = (cs_padj.T < 0.05).iloc[:, 0]
cs_acts = cs_acts.loc[:, msk]

cs_acts

In [ ]:
# Combine re_acts and re_padj
if(len(cs_acts.columns) > 0):
    combined_df = pd.DataFrame({
        'pathway': cs_acts.columns,
        'score': cs_acts.values.flatten(),
        'padj': cs_padj[cs_acts.columns].values.flatten()
    })
else:
    combined_df = pd.DataFrame(columns=['pathway','score','padj'])

In [ ]:
if(len(cs_acts.columns) > 0):
    dc.pl.barplot(data=cs_acts, name="AFRHigh_vs_AFRLow", figsize=(8, min(len(cs_acts.columns) * 3/10, 50*3/10)))
else:
    print("No significant CytoSig pathways found")

In [ ]:
if(len(cs_acts.columns) > 0):
    # Transform to df
    df = cs_acts.melt(value_name="score").merge(
        cs_padj.melt(value_name="padj")
        .assign(logpval=lambda x: x["padj"].clip(2.22e-4, 1))
        .assign(logpval=lambda x: -np.log10(x["logpval"]))
    )
    dc.pl.dotplot(df=df, top=50, x="score", y="variable", s="logpval", c="score", scale=1, figsize=(5, min(len(cs_acts.columns) * 3/10, 50*3/10)))
else:
    print("No significant progeny pathways found")

In [ ]:
#dc.pl.source_targets(data=merged_df, x="weight", y="stat", net=progeny, name="TNFa", top=15, figsize=(4, 4))

if(len(cs_acts.columns) > 0):
    
    n_pathways = len(cs_acts.columns)
    if n_pathways > 1:
        n_cols = 4
        n_rows = (n_pathways + n_cols - 1) // n_cols  # Ceiling division
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
        axes = axes.flatten()
        
        for idx, pathway in enumerate(cs_acts.columns.tolist()):
            print(pathway)
            dc.pl.source_targets(
                data=merged_df,
                x="weight",
                y="stat",
                net=cyto_sig,
                name=pathway,
                top=30,
                ax=axes[idx]
            )
            axes[idx].set_title(pathway, fontsize=10)
        
        # Hide empty subplots
        for idx in range(n_pathways, len(axes)):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()
    elif(n_pathways==1):
        for pathway in pw_acts.columns.tolist():
            print(pathway)
            dc.pl.source_targets(
                data=merged_df,
                x="weight",
                y="stat",
                net=cyto_sig,
                name=pathway,
                top=30,
                figsize=(4,4)
            )


else:
    print("No significant cyto_sig pathways found")


In [ ]:
if(len(cs_acts.columns) > 0):
    pos_leading_edge_dict = {}
    neg_leading_edge_dict = {}
    
    n_pathways = len(cs_acts.columns)

    for pathway in cs_acts.columns.tolist():
        _, pos_le = dc.pl.leading_edge(
            merged_df,
            stat="stat",
            net=cyto_sig[cyto_sig["weight"] > 0],
            name=pathway,
        )
        pos_leading_edge_dict[pathway] = ', '.join(pos_le)
        print("(+) leading edge:", pos_le[:5])
        
        _, neg_le = dc.pl.leading_edge(
            merged_df,
            stat="stat",
            net=cyto_sig[cyto_sig["weight"] < 0],
            name=pathway,
        )
        print("(-) leading edge:", neg_le[:5])
        neg_leading_edge_dict[pathway] = ', '.join(neg_le)
    
    combined_df['POSITIVE_leading_edge_genes'] = combined_df['pathway'].map(pos_leading_edge_dict)
    combined_df['NEGATIVE_leading_edge_genes'] = combined_df['pathway'].map(neg_leading_edge_dict)

else:
    print("No significant cyto_sig pathways found")


In [ ]:
combined_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_cytosig_pathways_score_fdr_legenes.tsv", sep='\t')
combined_df

CytoSig Alternative - Per-Sample ULM Analysis

In [ ]:
dc.mt.ulm(data=pdata_filt, net=cyto_sig, layer='normalized')

In [ ]:
pdata_filt.obsm['score_ulm']

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm

# Your model from earlier
model_formula = "cytokine_score ~ 1 + ANCESTRY_AFR + ANCESTRY_EAS + ANCESTRY_PEL + d_dx_amm_age + d_dx_amm_bmi + d_pt_sex + CGS_risk"

# Fit for each cytokine (column in score_ulm)
results = {}
for cytokine in pdata_filt.obsm['score_ulm'].columns:
    # Add cytokine score to obs for modeling
    pdata_filt.obs['cytokine_score'] = pdata_filt.obsm['score_ulm'][cytokine]
    
    # Fit model using formula API (R-like syntax)
    model = smf.ols(model_formula, data=pdata_filt.obs).fit()
    
    # Extract coefficient and p-value for ANCESTRY_AFR
    coef = model.params['ANCESTRY_AFR']
    pval = model.pvalues['ANCESTRY_AFR']
    
    results[cytokine] = {
        'coefficient': coef,
        'p_value': pval,
        'std_err': model.bse['ANCESTRY_AFR'],
        't_stat': model.tvalues['ANCESTRY_AFR']
    }

In [ ]:
from statsmodels.stats.multitest import multipletests

results_df = pd.DataFrame.from_dict(results, orient='index')
results_df = results_df.sort_values(by='p_value')

# Apply Benjamini-Hochberg FDR correction
results_df['fdr_p_value'] = multipletests(results_df['p_value'], method='fdr_bh')[1]

# Save CytoSig results to CSV
results_df.to_csv(output_dir_fig_tables + f"{FILE_NAME}_cytosig_ulm_regression_results.csv")

print(f"Saved CytoSig regression results to: {output_dir_fig_tables}{FILE_NAME}_cytosig_ulm_regression_results.csv")

# Save CytoSig ULM scores to CSV
cytosig_ulm_scores = pd.DataFrame(pdata_filt.obsm['score_ulm'], index=pdata_filt.obs_names)
cytosig_ulm_scores.to_csv(output_dir_fig_tables + f"{FILE_NAME}_cytosig_ulm_scores.csv")

print(f"Saved CytoSig ULM scores to: {output_dir_fig_tables}{FILE_NAME}_cytosig_ulm_scores.csv")

results_df[results_df['p_value'] < 0.1]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Define color palettes
color_AFR_high = "#86325F"
color_AFR_low = "#5A90C7"
fill_AFR_high = "#9b4472"
fill_AFR_low = "#699ed4"

# Filter for significant cytokines
sig_cytokines = results_df[results_df['fdr_p_value'] < 0.05].sort_values(by='coefficient').index.tolist()

print(f"Found {len(sig_cytokines)} significant cytokines (FDR p < 0.05)")

if len(sig_cytokines) > 0:
    n_cytokines = len(sig_cytokines)
    n_cols = 4
    n_rows = (n_cytokines + n_cols - 1) // n_cols  # Ceiling division
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes = axes.flatten()
    
    for idx, cytokine in enumerate(sig_cytokines):
        ax = axes[idx]
        
        # Get ULM scores for this cytokine
        cytokine_scores = pdata_filt.obsm['score_ulm'][cytokine]
        
        # Create a temporary dataframe for plotting
        plot_df = pd.DataFrame({
            'ULM_Score': cytokine_scores.values,
            'AFR_BINNED': pdata_filt.obs['AFR_BINNED'].values
        })
        
        # Reorder the categories so AFR High is on the right
        plot_df['AFR_BINNED'] = pd.Categorical(plot_df['AFR_BINNED'], 
                                                categories=['AFR Low', 'AFR High'], 
                                                ordered=True)
        
        # Get the p-value from results_df
        p_val = results_df.loc[cytokine, 'p_value']

        # Define color palettes for the two groups
        violin_colors = [fill_AFR_low, fill_AFR_high]
        box_colors = [color_AFR_low, color_AFR_high]
        
        # Plot violin plot with transparency
        parts = ax.violinplot([plot_df[plot_df['AFR_BINNED'] == 'AFR Low']['ULM_Score'].values,
                               plot_df[plot_df['AFR_BINNED'] == 'AFR High']['ULM_Score'].values],
                              positions=[0, 1],
                              showmeans=False, 
                              showmedians=False,
                              showextrema=False)
        
        # Color the violin plots with transparency
        for i, pc in enumerate(parts['bodies']):
            pc.set_facecolor(violin_colors[i])
            pc.set_alpha(0.4)
            pc.set_edgecolor('black')
            pc.set_linewidth(1)
        
        # Overlay box plot with custom colors
        bp = ax.boxplot([plot_df[plot_df['AFR_BINNED'] == 'AFR Low']['ULM_Score'].values,
                         plot_df[plot_df['AFR_BINNED'] == 'AFR High']['ULM_Score'].values],
                        positions=[0, 1],
                        widths=0.3,
                        patch_artist=True,
                        showfliers=True)
        
        # Color the box plots
        for i, (patch, color) in enumerate(zip(bp['boxes'], box_colors)):
            patch.set_facecolor(color)
            patch.set_alpha(0.8)
        
        # Color the other box plot elements
        for i, color in enumerate(box_colors):
            bp['medians'][i].set_color('black')
            bp['medians'][i].set_linewidth(2)
            bp['whiskers'][i*2].set_color(color)
            bp['whiskers'][i*2+1].set_color(color)
            bp['caps'][i*2].set_color(color)
            bp['caps'][i*2+1].set_color(color)
            if bp['fliers'][i].get_data()[0].size > 0:
                bp['fliers'][i].set_markerfacecolor(color)
                bp['fliers'][i].set_markeredgecolor(color)
                bp['fliers'][i].set_alpha(0.5)
        
        # Add individual points with matching colors
        for i, category in enumerate(['AFR Low', 'AFR High']):
            y_data = plot_df[plot_df['AFR_BINNED'] == category]['ULM_Score'].values
            x_data = np.random.normal(i, 0.04, size=len(y_data))
            ax.scatter(x_data, y_data, 
                      color=box_colors[i], 
                      alpha=0.3, 
                      s=30,
                      edgecolors='none')
        
        # Calculate position for p-value annotation and expand ylim
        y_max = plot_df['ULM_Score'].max()
        y_min = plot_df['ULM_Score'].min()
        y_range = y_max - y_min
        
        # Position p-value text 5% above the max value
        p_val_y = y_max + 0.05 * y_range
        
        # Expand y-axis to accommodate p-value text (add 12% above max to ensure text fits)
        ax.set_ylim(y_min - 0.05 * y_range, y_max + 0.15 * y_range)
        
        # Format p-value: 3 sig figs if >= 0.001, scientific notation if < 0.001
        if p_val >= 0.001:
            p_val_text = f'p = {p_val:.2g}'
        else:
            p_val_text = f'p = {p_val:.2e}'
        
        ax.text(0.5, p_val_y, 
               p_val_text, 
               ha='center', va='bottom', fontsize=12, fontweight='bold')
        
        # Set labels and title
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['AFR Low', 'AFR High'])
        ax.set_xlabel('African Genetic Similarity', fontsize=12)
        ax.set_ylabel('ULM Score', fontsize=12)
        ax.set_title(f'{cytokine}', fontsize=14, fontweight='bold')
        
    for idx in range(n_cytokines, len(axes)):
        axes[idx].axis('off')    
    plt.tight_layout()
    plt.show()
else:
    print("No significant cytokines to plot")

<h1> Gene expression survival analyses </h1>

In [ ]:
#papermill_description=Survival

import sys
n_pat = pdata_filt.n_obs
if(n_pat < 23):
    print(f"Too few patient for survival analyses")
    sys.exit(0)
else:
    print(f"Survival analyses can continue")


In [ ]:
# Convert survival columns to float to avoid categorical issues
pdata_filt.obs['ttcpfs'] = pd.to_numeric(pdata.obs['ttcpfs'], errors='coerce')
pdata_filt.obs['censpfs'] = pd.to_numeric(pdata.obs['censpfs'], errors='coerce')

print("Converted survival columns to numeric:")
print(f"ttcpfs dtype: {pdata_filt.obs['ttcpfs'].dtype}")
print(f"censpfs dtype: {pdata_filt.obs['censpfs'].dtype}")
print(f"ttcpfs range: {pdata_filt.obs['ttcpfs'].min():.1f} - {pdata_filt.obs['ttcpfs'].max():.1f}")
print(f"censpfs values: {pdata_filt.obs['censpfs'].unique()}")

In [ ]:
print(os.getcwd())
os.chdir('/workspaces/MMRF_Emory_WP/')
print(os.getcwd())


In [ ]:
pdata_filt.obs['d_tx_induction_cat'].value_counts()


In [ ]:
pdata_filt.obs['d_tx_induction_cat'].loc[pdata_filt.obs['d_tx_induction_cat'] == 'chemo_imid_pi_steroid'] = 'imid_pi_steroid' 
pdata_filt.obs['d_tx_induction_cat'].loc[pdata_filt.obs['d_tx_induction_cat'] == 'chemo_imid_steroid'] = 'imid_steroid' 
pdata_filt.obs['d_tx_induction_cat'].loc[pdata_filt.obs['d_tx_induction_cat'] == 'chemo_pi_steroid'] = 'pi_steroid' 
pdata_filt.obs['d_tx_induction_cat'] = pdata_filt.obs['d_tx_induction_cat'].cat.remove_unused_categories()
pdata_filt.obs['d_tx_induction_cat'].value_counts()


In [ ]:
#papermill_description=COX_model_study_site

import importlib
from source.Helper_Functions import general_python as wp_scanpy
importlib.reload(wp_scanpy)

# Check if results already exist and should be reloaded
cox_site_results_file = output_dir_fig_tables + f"{FILE_NAME}_cox_regression_d_dx_amm_age_only_model_results.csv"
if RELOAD_IF_EXISTING and os.path.exists(cox_site_results_file):
    print(f"Loading existing Cox regression (d_dx_amm_age model) results from: {cox_site_results_file}")
    test_results_site = pd.read_csv(cox_site_results_file, index_col=0)
    print(f"Loaded {len(test_results_site)} genes from existing results")
else:
    print("=== Running Cox regression analysis with all genes (d_dx_amm_age adjustment) ===")
    test_results_site = wp_scanpy.genome_wide_cox_analysis(
        pdata_filt, 
        layer='scaled',
        covariates=['d_dx_amm_age'],
        max_genes=None,
        verbose=True
    )
    
    test_results_site["model"] = "d_dx_amm_age"
    test_results_site["survival_type"] = "PFS"
    test_results_site.to_csv(cox_site_results_file)

# Display results
print("Top 10 results (by p-value):")
top_results = test_results_site[test_results_site['status'] == 'success'].nsmallest(30, 'p_value')
print(top_results[['gene', 'hazard_ratio', 'p_value', 'fdr_p_value', 'n_patients', 'n_events']])

if (test_results_site['p_value'] < 0.05).any():
    wp_scanpy.plot_cox_volcano(test_results_site, 
                    title=f"{COMPARTMENT_DESCRIPTION}: Cox Regression: Gene Expression vs Survival (d_dx_amm_age Adjusted) - Raw P-Values",
                    p_val_column='p_value',
                    p_threshold=0.05, 
                    top_n_labels=75)
else:
    print("No significant genes found at p < 0.05 - Skipping Plot")

if (test_results_site['fdr_p_value'] < 0.05).any():
    wp_scanpy.plot_cox_volcano(test_results_site, 
                    title=f"{COMPARTMENT_DESCRIPTION}: Cox Regression: Gene Expression vs Survival (d_dx_amm_age Adjusted) - FDR Corrected",
                    p_val_column='fdr_p_value',
                    p_threshold=0.05, 
                    top_n_labels=75)
else:
    print("No significant genes found at FDR padj < 0.05 - Skipping Plot")


In [ ]:
#papermill_description=COX_model_full

import importlib
from source.Helper_Functions import general_python as wp_scanpy
importlib.reload(wp_scanpy)

# Check if results already exist and should be reloaded
cox_full_results_file = output_dir_fig_tables + f"{FILE_NAME}_cox_regression_full_model_results.csv"
if RELOAD_IF_EXISTING and os.path.exists(cox_full_results_file):
    print(f"Loading existing Cox regression (full model) results from: {cox_full_results_file}")
    test_results_full = pd.read_csv(cox_full_results_file, index_col=0)
    print(f"Loaded {len(test_results_full)} genes from existing results")
else:
    print("=== Running Cox regression analysis with all genes (Full Model) ===")
    test_results_full = wp_scanpy.genome_wide_cox_analysis(
        pdata_filt, 
        layer='scaled',
        covariates=['ANCESTRY.AFR','ANCESTRY.PEL','ANCESTRY.EAS','d_dx_amm_age','d_dx_amm_bmi','CGS_risk', 'd_amm_tx_asct_1st', 'd_pt_sex'],
        max_genes=None,
        verbose=True
    )
    
    test_results_full["model"] = "ANCESTRY.AFR + ANCESTRY.PEL + ANCESTRY.EAS + d_dx_amm_age + d_dx_amm_bmi + CGS_risk + d_amm_tx_asct_1st + d_pt_sex"
    test_results_full["survival_type"] = "PFS"
    test_results_full.to_csv(cox_full_results_file)

# Display results
print("Top 10 results (by p-value):")
top_results = test_results_full[test_results_full['status'] == 'success'].nsmallest(30, 'p_value')
print(top_results[['gene', 'hazard_ratio', 'p_value', 'fdr_p_value', 'n_patients', 'n_events']])

if (test_results_full['p_value'] < 0.05).any():
    wp_scanpy.plot_cox_volcano(test_results_full, 
                    title=f"{COMPARTMENT_DESCRIPTION}: Cox Regression: Gene Expression vs Survival (Full Model) - Raw P-Values",
                    p_val_column='p_value',
                    p_threshold=0.05, 
                    top_n_labels=75)
else:
    print("No significant genes found at p < 0.05 - Skipping Plot")

if (test_results_full['fdr_p_value'] < 0.05).any():
    wp_scanpy.plot_cox_volcano(test_results_full, 
                    title=f"{COMPARTMENT_DESCRIPTION}: Cox Regression: Gene Expression vs Survival (Full Model) - FDR Corrected",
                    p_val_column='fdr_p_value',
                    p_threshold=0.05, 
                    top_n_labels=75)
else:
    print("No significant genes found at FDR padj < 0.05 - Skipping Plot")


Comparing shrunk log2FC vs cox p-value (Study site adjusted model). 

Using raw p-value for the cox model, adjusted for DEG (this is not a final visual - internal only)

In [ ]:
import importlib
from source.Helper_Functions import general_python as wp_scanpy
importlib.reload(wp_scanpy)

merged_data_labeled = wp_scanpy.create_survival_vs_deg_scatterplot_with_labels(
    test_results_site, 
    merged_df,
    cox_pval_column='p_value',
    deg_pval_column='padj',
    cox_p_threshold=0.05,
    deg_p_threshold=0.05,
    figsize=(14, 10),
    label_both_sig=True,
    compartment_name=COMPARTMENT_DESCRIPTION
)

Comparing shrunk log2FC vs cox p-value (Full Model)

In [ ]:
merged_data_labeled = wp_scanpy.create_survival_vs_deg_scatterplot_with_labels(
    test_results_full, 
    merged_df,
    cox_pval_column='p_value',
    deg_pval_column='padj',
    cox_p_threshold=0.05,
    deg_p_threshold=0.05,
    figsize=(14, 10),
    label_both_sig=True,
    compartment_name=COMPARTMENT_DESCRIPTION
)

<h3> Outcome Associated Pathways via Decoupler </h3>

Repeating the decoupler analyses, but basically doing it by sign(log(HR)) * -log(p-val) from the adjusted coxPH model.

In [ ]:
test_results_full = test_results_full.set_index('gene')
test_results_full

In [ ]:
test_results_full[["hazard_ratio_log10"]] = test_results_full[["hazard_ratio"]].apply(np.log2)

test_results_full["signed_pval"] = test_results_full["p_value"].apply(np.log2).multiply(-1).multiply(test_results_full["hazard_ratio_log10"].apply(np.sign))


data = test_results_full[["p_value"]].T.rename(index={"p_value": "AFRHigh_vs_AFRLow"}).apply(np.log2).multiply(-1)
data

dataHR = test_results_full[["hazard_ratio"]].T.rename(index={"hazard_ratio": "AFRHigh_vs_AFRLow"}).apply(np.log2)
dataHR

test_results_full


In [ ]:
data_mult = data.multiply(dataHR.apply(np.sign))

data_mult

<h3> CoxPH - Decoupler TF (collectri) </h3>

In [ ]:
#papermill_description=COX_model_pathways

# Run
tf_acts, tf_padj = dc.mt.ulm(data=data_mult, net=collectri)

# Filter by sign padj
msk = (tf_padj.T < 0.05).iloc[:, 0]
tf_acts = tf_acts.loc[:, msk]

tf_acts


In [ ]:
if(len(tf_acts.columns) > 0):
    dc.pl.barplot(data=tf_acts, name="AFRHigh_vs_AFRLow", figsize=(4, 3))
else:
    print("No significant TFs found")
   
if(len(tf_acts.columns) > 0):
    dc.pl.network(
        net=collectri,
        data=data,
        score=tf_acts,
        sources=tf_acts.columns.tolist(),
        targets=10,
        figsize=(12, 7),
        vcenter=True,
        by_abs=True,
        size_node=30,
    )
else:
    print("No significant TFs found")

if(len(tf_acts.columns) > 0):
    
    n_tfs = len(tf_acts.columns)
    if n_tfs > 1:
        n_cols = 4
        n_rows = (n_tfs + n_cols - 1) // n_cols  # Ceiling division
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
        axes = axes.flatten()
        
        for idx, tf in enumerate(tf_acts.columns.tolist()):
            dc.pl.volcano(
                data=test_results_full,
                x="hazard_ratio_log10",
                y="p_value",
                net=collectri,
                name=tf,
                top=10,
                thr_stat=0.08,
                thr_sign=0.05,
                ax=axes[idx]
            )
            axes[idx].set_title(tf, fontsize=10)
        
        # Hide empty subplots
        for idx in range(n_tfs, len(axes)):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()
    elif(n_tfs==1):
        for tf in tf_acts.columns.tolist():
            dc.pl.volcano(
                data=test_results_full,
                x="hazard_ratio_log10",
                y="p_value",
                net=collectri,
                name=tf,
                top=10,
                thr_stat=0.08,
                thr_sign=0.05
            )


else:
    print("No significant TFs found")


<h3> CoxPH - Progeny </h3>

In [ ]:
# Run
pw_acts, pw_padj = dc.mt.ulm(data=data_mult, net=progeny)

# Filter by sign padj
msk = (pw_padj.T < 0.05).iloc[:, 0]
pw_acts = pw_acts.loc[:, msk]

print(pw_acts.head())

if(len(pw_acts.columns) > 0):
    dc.pl.barplot(data=pw_acts, name="AFRHigh_vs_AFRLow", figsize=(3, 3))
else:
    print("No significant progeny pathways found")

if(len(pw_acts.columns) > 0):
    # Transform to df
    df = pw_acts.melt(value_name="score").merge(
        pw_padj.melt(value_name="padj")
        .assign(logpval=lambda x: x["padj"].clip(2.22e-4, 1))
        .assign(logpval=lambda x: -np.log10(x["logpval"]))
    )
    dc.pl.dotplot(df=df, x="score", y="variable", s="logpval", c="score", scale=1, figsize=(4, 4), top=30)
else:
    print("No significant progeny pathways found")

#dc.pl.source_targets(data=merged_df, x="weight", y="stat", net=progeny, name="TNFa", top=15, figsize=(4, 4))

if(len(pw_acts.columns) > 0):
    
    n_pathways = len(pw_acts.columns)
    if n_pathways > 1:
        n_cols = 2  # Two columns for progeny (positive and negative leading edge)
        n_rows = (n_pathways + n_cols - 1) // n_cols  # Ceiling division
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
        axes = axes.flatten()
        
        for idx, pathway in enumerate(pw_acts.columns.tolist()):
            print(pathway)
            dc.pl.source_targets(
                data=test_results_full,
                x="weight",
                y="hazard_ratio_log10",
                net=progeny,
                name=pathway,
                top=20,
                ax=axes[idx]
            )
            axes[idx].set_title(pathway, fontsize=10)
        
        # Hide empty subplots
        for idx in range(n_pathways, len(axes)):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()
    elif(n_pathways==1):
        for pathway in pw_acts.columns.tolist():
            print(pathway)
            dc.pl.source_targets(
                data=test_results_full,
                x="weight",
                y="hazard_ratio_log10",
                net=progeny,
                name=pathway,
                top=20,
                figsize=(4,4)
            )


else:
    print("No significant progeny pathways found")

if(len(pw_acts.columns) > 0):
    for pathway in pw_acts.columns.tolist():
        _, pos_le = dc.pl.leading_edge(
            test_results_full,
            stat="signed_pval",
            net=progeny[progeny["weight"] > 0],
            name=pathway,
        )
        print("(+) leading edge:", pos_le[:5])
        _, neg_le = dc.pl.leading_edge(
            test_results_full,
            stat="signed_pval",
            net=progeny[progeny["weight"] < 0],
            name=pathway,
        )
        print("(-) leading edge:", neg_le[:5])
else:
    print("No significant progeny pathways found")


<h3> Reactome CoxPH </h3>

In [ ]:
# Run
re_acts, re_padj = dc.mt.ulm(data=data_mult, net=reactome_pathways_df)

# Filter by sign padj
msk = (re_padj.T < 0.05).iloc[:, 0]
re_acts = re_acts.loc[:, msk]

re_acts

In [ ]:
if(len(re_acts.columns) > 0):
    dc.pl.barplot(data=re_acts, name="AFRHigh_vs_AFRLow", figsize=(10, min(len(re_acts.columns) * 3/5, 50*3/5)), top=50)
else:
    print("No significant reactome pathways found")

In [ ]:
if(len(re_acts.columns) > 0):
    for pathway in re_acts.columns.tolist():
        _, le = dc.pl.leading_edge(
            test_results_full,
            stat="signed_pval",
            net=reactome_pathways_df,
            name=pathway,
        )
        print(f"{pathway} leading edge:", le[:10])
else:
    print("No significant reactome pathways found")

In [ ]:
if(len(re_acts.columns) > 0):
    n_pathways = len(re_acts.columns)
    if(n_pathways > 1):
        n_cols = 4
        n_rows = (n_pathways + n_cols - 1) // n_cols  # Ceiling division
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
        axes = axes.flatten()
        
        for idx, pathway in enumerate(re_acts.columns.tolist()):
            dc.pl.volcano(
                data=test_results_full,
                x="hazard_ratio_log10",
                y="p_value",
                net=reactome_pathways_df,
                name=pathway,
                top=20,
                thr_stat=0.08,
                thr_sign=0.05,
                ax=axes[idx]
            )
            axes[idx].set_title(pathway, fontsize=10)
        
        # Hide empty subplots
        for idx in range(n_pathways, len(axes)):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()
    elif(n_pathways==1):
        for pathway in re_acts.columns.tolist():
            dc.pl.volcano(
                data=merged_df,
                x="log2FoldChange",
                y="padj",
                net=reactome_pathways_df,
                name=pathway,
                top=30,
                thr_stat=0.1,
                thr_sign=0.1
            )
else:
    print("No significant reactome pathways found")

<h3> CoxPH - Hallmark </h3>

In [ ]:
# Run
hm_acts, hm_padj = dc.mt.ulm(data=data_mult, net=hallmark)

# Filter by sign padj
msk = (hm_padj.T < 0.05).iloc[:, 0]
hm_acts = hm_acts.loc[:, msk]

hm_acts

In [ ]:
if(len(hm_acts.columns) > 0):
    dc.pl.barplot(data=hm_acts, name="AFRHigh_vs_AFRLow", figsize=(10, min(len(hm_acts.columns) * 3/5, 50*3/5)), top=50)
else:
    print("No significant hallmark pathways found")

if(len(hm_acts.columns) > 0):
    for pathway in hm_acts.columns.tolist():
        _, le = dc.pl.leading_edge(
            test_results_full,
            stat="signed_pval",
            net=hallmark,
            name=pathway,
        )
        print(f"{pathway} leading edge:", le[:10])
else:
    print("No significant hallmark pathways found")


if(len(hm_acts.columns) > 0):
    for pathway in hm_acts.columns.tolist():
        dc.pl.volcano(
            data=test_results_full,
            x="hazard_ratio_log10",
            y="p_value",
            net=hallmark,
            name=pathway,
            top=10,
            thr_stat=0.08,
            thr_sign=0.05
        )
else:
    print("No significant hallmark pathways found")

<h3> CoxPH - Custom Pathways </h3>

In [ ]:
# Run
custom_acts, custom_padj = dc.mt.ulm(data=data_mult, net=custom_pathways_df)

# Filter by sign padj
msk = (custom_padj.T < 0.05).iloc[:, 0]
custom_acts = custom_acts.loc[:, msk]

custom_acts

In [ ]:

if(len(custom_acts.columns) > 0):
    dc.pl.barplot(data=custom_acts, name="AFRHigh_vs_AFRLow", figsize=(6, 3))
else:
    print("No significant hallmark pathways found")

if(len(custom_acts.columns) > 0):
    for pathway in custom_acts.columns.tolist():
        _, le = dc.pl.leading_edge(
            test_results_full,
            stat="signed_pval",
            net=custom_pathways_df,
            name=pathway,
        )
        print(f"{pathway} leading edge:", le[:10])
else:
    print("No significant custom pathways found")


if(len(custom_acts.columns) > 0):
    for pathway in custom_acts.columns.tolist():
        dc.pl.volcano(
            data=test_results_full,
            x="hazard_ratio_log10",
            y="p_value",
            net=custom_pathways_df,
            name=pathway,
            top=10,
            thr_stat=0.08,
            thr_sign=0.05
        )
else:
    print("No significant custom pathways found")

<h1> !! EXPERIMENTAL ANALYSES !! - ROBUSTLY INSPECT BEFORE INCLUDING </h1>
<h2> Stratified Cox Analysis: Comparing High vs Low African Ancestry Groups </h2>

Running Cox regression separately for patients with high AFR ancestry (>0.5) and low AFR ancestry (<=0.5) to identify genes with differential survival associations between groups.

NOTE - EXPERIMENTAL EXPLORATORY ANALYSIS. DUE TO LOWER NUMBER OF PATIENTS IN HIGH AFR GROUP WE LIKELY RUN INTO ISSUES WITH COXPH ASSUMPTIONS. TAKE WITH A GRAIN OF SALT. IF ANYTHING STANDS OUT, PERFORM MORE ROBUST SECONDARY ANALYSIS

In [ ]:
#papermill_description=COX_model_stratified

import importlib
from source.Helper_Functions import general_python as wp_scanpy
importlib.reload(wp_scanpy)

# Check if results already exist and should be reloaded
cox_stratified_results_file = output_dir_fig_tables + f"{FILE_NAME}_cox_regression_stratified_by_AFR.csv"

n_afr_high = pdata_filt[pdata_filt.obs['ANCESTRY.AFR'] > 0.5].n_obs
n_afr_low = pdata_filt[pdata_filt.obs['ANCESTRY.AFR'] <= 0.5].n_obs

if(n_afr_high < 10 or n_afr_low < 10):
    print(f"Not enough samples in one of the AFR groups to run stratified analysis (AFR High: {n_afr_high}, AFR Low: {n_afr_low})")
    stratified_results = pd.DataFrame()
else:
    if RELOAD_IF_EXISTING and os.path.exists(cox_stratified_results_file):
        print(f"Loading existing stratified Cox regression results from: {cox_stratified_results_file}")
        stratified_results = pd.read_csv(cox_stratified_results_file, index_col=0)
        print(f"Loaded stratified results for {len(stratified_results)} genes")
    else:
        print("=== Running stratified Cox regression analysis ===")
        
        # Subset to high AFR ancestry (>0.5)
        pdata_filt_high = pdata_filt[pdata_filt.obs['ANCESTRY.AFR'] > 0.5].copy()
        print(f"High AFR group: {pdata_filt_high.n_obs} samples")
        
        # Separately re-scale for each group
        sc.pp.normalize_total(pdata_filt_high, target_sum=1e4)
        sc.pp.log1p(pdata_filt_high) # Ignore warning - .uns log1p will be set from subsetting
        pdata_filt_high.layers["normalized"] = pdata_filt_high.X.copy()
        sc.pp.scale(pdata_filt_high, max_value=10)
        pdata_filt_high.layers["scaled"] = pdata_filt_high.X.copy()
        dc.pp.swap_layer(adata=pdata_filt_high, key="counts", inplace=True)


        # Subset to low AFR ancestry (<=0.5)
        pdata_filt_low = pdata_filt[pdata_filt.obs['ANCESTRY.AFR'] <= 0.5].copy()
        print(f"Low AFR group: {pdata_filt_low.n_obs} samples")
        # Separately re-scale for each group
        sc.pp.normalize_total(pdata_filt_low, target_sum=1e4)
        sc.pp.log1p(pdata_filt_low)
        pdata_filt_low.layers["normalized"] = pdata_filt_low.X.copy()
        sc.pp.scale(pdata_filt_low, max_value=10)
        pdata_filt_low.layers["scaled"] = pdata_filt_low.X.copy()
        dc.pp.swap_layer(adata=pdata_filt_low, key="counts", inplace=True)



        # Run Cox analysis on high AFR group
        print("\n--- Running Cox analysis on HIGH AFR group ---")
        test_results_high = wp_scanpy.genome_wide_cox_analysis(
            pdata_filt_high, 
            layer='scaled',
            covariates=['d_dx_amm_age','d_dx_amm_bmi','CGS_risk', 'd_tx_induction_cat', 'd_amm_tx_asct_1st', 'd_pt_sex'],
            max_genes=None,
            verbose=True
        )
        
        # Run Cox analysis on low AFR group
        print("\n--- Running Cox analysis on LOW AFR group ---")
        test_results_low = wp_scanpy.genome_wide_cox_analysis(
            pdata_filt_low, 
            layer='scaled',
            covariates=['d_dx_amm_age','d_dx_amm_bmi','CGS_risk', 'd_tx_induction_cat', 'd_amm_tx_asct_1st', 'd_pt_sex'],
            max_genes=None,
            verbose=True
        )
        
        # Prefix columns and merge
        test_results_high = test_results_high.add_prefix('AFRHIGH_')
        test_results_low = test_results_low.add_prefix('AFRLOW_')
        
        # Merge on gene names
        stratified_results = pd.merge(
            test_results_high, 
            test_results_low, 
            left_on='AFRHIGH_gene', 
            right_on='AFRLOW_gene', 
            how='outer'
        )
        
        # Clean up gene column
        stratified_results['gene'] = stratified_results['AFRHIGH_gene'].fillna(stratified_results['AFRLOW_gene'])
        stratified_results = stratified_results.set_index('gene')
        
        # Save results
        stratified_results.to_csv(cox_stratified_results_file)
        print(f"\nSaved stratified results to {cox_stratified_results_file}")

# Display summary
    print("\nStratified Cox Regression Summary:")
    print(f"Total genes analyzed: {len(stratified_results)}")
    print(f"Significant in HIGH AFR (p<0.05): {(stratified_results['AFRHIGH_p_value'] < 0.05).sum()}")
    print(f"Significant in LOW AFR (p<0.05): {(stratified_results['AFRLOW_p_value'] < 0.05).sum()}")
    print(f"Significant in BOTH (p<0.05): {((stratified_results['AFRHIGH_p_value'] < 0.05) & (stratified_results['AFRLOW_p_value'] < 0.05)).sum()}")

stratified_results.head()

In [ ]:
if(n_afr_high < 10 or n_afr_low < 10):
    print("Not enough samples in one of the AFR groups to plot stratified results")
else:
    # Import adjust_text for label spacing
    from adjustText import adjust_text

    # Calculate log10 hazard ratios for plotting
    stratified_results['AFRHIGH_log10_hr'] = np.log10(stratified_results['AFRHIGH_hazard_ratio'])
    stratified_results['AFRLOW_log10_hr'] = np.log10(stratified_results['AFRLOW_hazard_ratio'])

    # Identify genes significant in both groups
    sig_in_both = (stratified_results['AFRHIGH_p_value'] < 0.05) & (stratified_results['AFRLOW_p_value'] < 0.05)
    sig_in_high_only = (stratified_results['AFRHIGH_p_value'] < 0.05) & (stratified_results['AFRLOW_p_value'] >= 0.05)
    sig_in_low_only = (stratified_results['AFRHIGH_p_value'] >= 0.05) & (stratified_results['AFRLOW_p_value'] < 0.05)

    # Create plot
    fig, ax = plt.subplots(figsize=(10, 10))

    # Plot non-significant genes
    mask_nonsig = ~sig_in_both & ~sig_in_high_only & ~sig_in_low_only
    ax.scatter(
        stratified_results.loc[mask_nonsig, 'AFRLOW_log10_hr'],
        stratified_results.loc[mask_nonsig, 'AFRHIGH_log10_hr'],
        alpha=0.3,
        s=20,
        color='lightgray',
        label='Not significant'
    )

    # Plot genes significant in high only
    if sig_in_high_only.sum() > 0:
        ax.scatter(
            stratified_results.loc[sig_in_high_only, 'AFRLOW_log10_hr'],
            stratified_results.loc[sig_in_high_only, 'AFRHIGH_log10_hr'],
            alpha=0.6,
            s=40,
            color='#86325F',
            label=f'Sig in HIGH AFR only (n={sig_in_high_only.sum()})'
        )

    # Plot genes significant in low only
    if sig_in_low_only.sum() > 0:
        ax.scatter(
            stratified_results.loc[sig_in_low_only, 'AFRLOW_log10_hr'],
            stratified_results.loc[sig_in_low_only, 'AFRHIGH_log10_hr'],
            alpha=0.6,
            s=40,
            color='#5A90C7',
            label=f'Sig in LOW AFR only (n={sig_in_low_only.sum()})'
        )

    # Plot genes significant in both
    if sig_in_both.sum() > 0:
        ax.scatter(
            stratified_results.loc[sig_in_both, 'AFRLOW_log10_hr'],
            stratified_results.loc[sig_in_both, 'AFRHIGH_log10_hr'],
            alpha=0.8,
            s=60,
            color='#D4AF37',
            edgecolors='black',
            linewidths=1,
            label=f'Sig in BOTH groups (n={sig_in_both.sum()})'
        )
        
        # Label genes significant in both using ax.text with white background
        texts = []
        for gene in stratified_results[sig_in_both].index:  
            text = ax.text(
                stratified_results.loc[gene, 'AFRLOW_log10_hr'],
                stratified_results.loc[gene, 'AFRHIGH_log10_hr'],
                gene,
                fontsize=8,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='none', alpha=0.8)
            )
            texts.append(text)
        
        # Use adjust_text to spread labels apart
        if texts:
            adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', lw=0.5, alpha=0.5))

    # Add diagonal reference line
    lims = [
        np.nanmin([ax.get_xlim()[0], ax.get_ylim()[0]]),
        np.nanmax([ax.get_xlim()[1], ax.get_ylim()[1]])
    ]
    ax.plot(lims, lims, 'k--', alpha=0.3, linewidth=1, label='y=x')

    # Add horizontal and vertical lines at 0
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.3)
    ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5, alpha=0.3)

    # Add corner annotations
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    offset_x = (xlim[1] - xlim[0]) * 0.02
    offset_y = (ylim[1] - ylim[0]) * 0.02

    # Top right: "Worse Outcomes (Both)"
    ax.text(xlim[1] - offset_x, ylim[1] - offset_y, "Worse Outcomes (Both)", 
            fontsize=9, ha='right', va='top', style='italic', alpha=0.7,
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.7))

    # Bottom left: "Better Outcomes (Both)"
    ax.text(xlim[0] + offset_x, ylim[0] + offset_y, "Better Outcomes (Both)", 
            fontsize=9, ha='left', va='bottom', style='italic', alpha=0.7,
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.7))

    # Bottom right: "Low AFR - Worse Outcomes\nHigh AFR - Better Outcomes"
    ax.text(xlim[1] - offset_x, ylim[0] + offset_y, "Low AFR - Worse Outcomes\nHigh AFR - Better Outcomes", 
            fontsize=9, ha='right', va='bottom', style='italic', alpha=0.7,
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.7))

    # Top left: "Low AFR - Better Outcomes\nHigh AFR - Worse Outcomes"
    ax.text(xlim[0] + offset_x, ylim[1] - offset_y, "Low AFR - Better Outcomes\nHigh AFR - Worse Outcomes", 
            fontsize=9, ha='left', va='top', style='italic', alpha=0.7,
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.7))

    ax.set_xlabel('Log10 Hazard Ratio (LOW AFR Group)', fontsize=12)
    ax.set_ylabel('Log10 Hazard Ratio (HIGH AFR Group)', fontsize=12)
    ax.set_title(f'{COMPARTMENT_DESCRIPTION}: Stratified Cox Regression\nComparing Hazard Ratios between AFR Groups', fontsize=14, fontweight='bold')
    ax.legend(loc='upper left', bbox_to_anchor=(0.02, 0.72), framealpha=0.9)
    ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

    # Detailed breakdown of genes significant in both groups
    if sig_in_both.sum() > 0:
        both_genes = stratified_results[sig_in_both][['AFRHIGH_hazard_ratio', 'AFRHIGH_p_value', 
                                                        'AFRLOW_hazard_ratio', 'AFRLOW_p_value']].copy()
        
        # Category 1: Poor outcomes in both groups (HR > 1 in both)
        poor_both = both_genes[(both_genes['AFRHIGH_hazard_ratio'] > 1) & (both_genes['AFRLOW_hazard_ratio'] > 1)]
        print("\n" + "="*80)
        print("CATEGORY 1: Worse Outcomes in BOTH Groups (HR > 1 in both)")
        print("="*80)
        print(f"Number of genes: {len(poor_both)}")
        if len(poor_both) > 0:
            print(poor_both.sort_values('AFRHIGH_p_value'))
        print("\n")
        
        # Category 2: Better outcomes in both groups (HR < 1 in both)
        better_both = both_genes[(both_genes['AFRHIGH_hazard_ratio'] < 1) & (both_genes['AFRLOW_hazard_ratio'] < 1)]
        print("="*80)
        print("CATEGORY 2: Better Outcomes in BOTH Groups (HR < 1 in both)")
        print("="*80)
        print(f"Number of genes: {len(better_both)}")
        if len(better_both) > 0:
            print(better_both.sort_values('AFRHIGH_p_value'))
        print("\n")
        
        # Category 3: Opposite effects - High AFR worse, Low AFR better (HR > 1 in High, HR < 1 in Low)
        high_worse_low_better = both_genes[(both_genes['AFRHIGH_hazard_ratio'] > 1) & (both_genes['AFRLOW_hazard_ratio'] <= 1)]
        print("="*80)
        print("CATEGORY 3: Opposite Effects - High AFR Worse, Low AFR Better")
        print("(HR > 1 in High AFR, HR < 1 in Low AFR)")
        print("="*80)
        print(f"Number of genes: {len(high_worse_low_better)}")
        if len(high_worse_low_better) > 0:
            print(high_worse_low_better.sort_values('AFRHIGH_p_value'))
        print("\n")
        
        # Category 4: Opposite effects - Low AFR worse, High AFR better (HR < 1 in High, HR > 1 in Low)
        low_worse_high_better = both_genes[(both_genes['AFRHIGH_hazard_ratio'] <= 1) & (both_genes['AFRLOW_hazard_ratio'] > 1)]
        print("="*80)
        print("CATEGORY 4: Opposite Effects - Low AFR Worse, High AFR Better")
        print("(HR < 1 in High AFR, HR > 1 in Low AFR)")
        print("="*80)
        print(f"Number of genes: {len(low_worse_high_better)}")
        if len(low_worse_high_better) > 0:
            print(low_worse_high_better.sort_values('AFRHIGH_p_value'))
        print("\n")
        
        # Summary
        print("="*80)
        print("SUMMARY")
        print("="*80)
        print(f"Total genes significant in both groups: {len(both_genes)}")
        print(f"  - Worse outcomes in both: {len(poor_both)}")
        print(f"  - Better outcomes in both: {len(better_both)}")
        print(f"  - High AFR worse, Low AFR better: {len(high_worse_low_better)}")
        print(f"  - Low AFR worse, High AFR better: {len(low_worse_high_better)}")
    else:
        print("\nNo genes significant in BOTH groups (p<0.05)")

<h3> Interaction Term Analyses </h3>

In [ ]:
#papermill_description=COX_model_interactions

import importlib
from source.Helper_Functions import general_python as wp_scanpy
importlib.reload(wp_scanpy)

# Check if results already exist and should be reloaded
cox_interaction_results_file = output_dir_fig_tables + f"{FILE_NAME}_cox_regression_interaction_AFR.csv"

if RELOAD_IF_EXISTING and os.path.exists(cox_interaction_results_file):
    print(f"Loading existing interaction Cox regression results from: {cox_interaction_results_file}")
    interaction_results = pd.read_csv(cox_interaction_results_file, index_col=0)
    print(f"Loaded interaction results for {len(interaction_results)} genes")
else:
    print("=== Running Cox regression with Gene × ANCESTRY.AFR interaction ===")
    print("This analysis tests whether the effect of gene expression on survival")
    print("differs between patients with high vs low African ancestry.\n")
    
    # Run Cox analysis with interaction term
    interaction_results = wp_scanpy.genome_wide_cox_analysis_with_interaction(
        pdata_filt, 
        interaction_covariate='ANCESTRY.AFR',
        layer='scaled',
        covariates=['d_dx_amm_age','d_dx_amm_bmi','CGS_risk', 'd_tx_induction_cat', 'd_amm_tx_asct_1st', 'd_pt_sex'],
        max_genes=None,
        verbose=True
    )
    
    # Save results
    interaction_results.to_csv(cox_interaction_results_file)
    print(f"\nSaved interaction results to {cox_interaction_results_file}")

# Display summary
print("\nInteraction Analysis Summary:")
print(f"Total genes analyzed: {len(interaction_results)}")
successful = (interaction_results['status'] == 'success').sum()
print(f"Successfully analyzed: {successful}")

if successful > 0:
    print(f"\nGene expression main effects (p<0.05): {(interaction_results['gene_p'] < 0.05).sum()}")
    print(f"Gene expression main effects (FDR<0.05): {(interaction_results['gene_fdr_p'] < 0.05).sum()}")
    
    print(f"\nANCESTRY.AFR main effects (p<0.05): {(interaction_results['covariate_p'] < 0.05).sum()}")
    print(f"ANCESTRY.AFR main effects (FDR<0.05): {(interaction_results['covariate_fdr_p'] < 0.05).sum()}")
    
    print(f"\nInteraction terms (p<0.05): {(interaction_results['interaction_p'] < 0.05).sum()}")
    print(f"Interaction terms (FDR<0.05): {(interaction_results['interaction_fdr_p'] < 0.05).sum()}")

interaction_results.head()


### Top Genes with Significant Interactions

Display genes where the interaction between gene expression and African ancestry is significant

In [ ]:
# Filter for successful analyses
interaction_success = interaction_results[interaction_results['status'] == 'success'].copy()

# Identify genes with significant interactions
sig_interaction = interaction_success[interaction_success['interaction_p'] < 0.05].copy()
sig_interaction_fdr = interaction_success[interaction_success['interaction_fdr_p'] < 0.05].copy()

print(f"Genes with nominal significant interactions (p<0.05): {len(sig_interaction)}")
print(f"Genes with FDR significant interactions (FDR<0.05): {len(sig_interaction_fdr)}")

if len(sig_interaction_fdr) > 0:
    print("\n=== TOP GENES WITH SIGNIFICANT GENE × ANCESTRY INTERACTIONS (FDR < 0.05) ===\n")
    
    # Sort by interaction p-value
    top_interactions = sig_interaction_fdr.sort_values('interaction_p')
    
    for i, (idx, row) in enumerate(top_interactions.head(20).iterrows(), 1):
        print(f"{i:2d}. {row['gene']:<15}")
        print(f"    Gene Expression HR: {row['gene_hr']:.3f} (p={row['gene_p']:.3e}, FDR={row['gene_fdr_p']:.3e})")
        print(f"    AFR Main Effect HR: {row['covariate_hr']:.3f} (p={row['covariate_p']:.3e})")
        print(f"    INTERACTION HR:     {row['interaction_hr']:.3f} (p={row['interaction_p']:.3e}, FDR={row['interaction_fdr_p']:.3e})")
        print()
        
elif len(sig_interaction) > 0:
    print("\n=== TOP GENES WITH NOMINAL SIGNIFICANT INTERACTIONS (p < 0.05) ===\n")
    
    # Sort by interaction p-value
    top_interactions = sig_interaction.sort_values('interaction_p')
    
    for i, (idx, row) in enumerate(top_interactions.head(20).iterrows(), 1):
        print(f"{i:2d}. {row['gene']:<15}")
        print(f"    Gene Expression HR: {row['gene_hr']:.3f} (p={row['gene_p']:.3e})")
        print(f"    AFR Main Effect HR: {row['covariate_hr']:.3f} (p={row['covariate_p']:.3e})")
        print(f"    INTERACTION HR:     {row['interaction_hr']:.3f} (p={row['interaction_p']:.3e})")
        print()
else:
    print("\nNo significant interactions found at p<0.05 threshold")

### Visualization: Gene Expression HR vs Interaction HR

This scatter plot shows the relationship between:
- **X-axis**: Gene expression main effect (hazard ratio)
- **Y-axis**: Interaction effect (how much AFR modulates the gene effect)

Genes in different quadrants have different interpretations:
- **Upper right**: Gene increases risk, and this effect is stronger in high AFR patients
- **Upper left**: Gene decreases risk, but less protective in high AFR patients  
- **Lower right**: Gene increases risk, but less so in high AFR patients
- **Lower left**: Gene decreases risk, and more protective in high AFR patients

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Prepare plot data
plot_df = interaction_success.copy()
plot_df['log2_gene_hr'] = np.log2(plot_df['gene_hr'])
plot_df['log2_interaction_hr'] = np.log2(plot_df['interaction_hr'])

# Define significance categories
plot_df['sig_category'] = 'Not significant'
plot_df.loc[(plot_df['gene_p'] < 0.05) & (plot_df['interaction_p'] >= 0.05), 'sig_category'] = 'Gene only'
plot_df.loc[(plot_df['gene_p'] >= 0.05) & (plot_df['interaction_p'] < 0.05), 'sig_category'] = 'Interaction only'
plot_df.loc[(plot_df['gene_p'] < 0.05) & (plot_df['interaction_p'] < 0.05), 'sig_category'] = 'Both significant'

# Count categories
category_counts = plot_df['sig_category'].value_counts()
print("Significance categories:")
for cat, count in category_counts.items():
    print(f"  {cat}: {count} genes")

# Create figure
fig, ax = plt.subplots(figsize=(14, 10))

# Define colors
colors = {
    'Not significant': '#CCCCCC',
    'Gene only': '#FF6B6B',
    'Interaction only': '#4ECDC4',
    'Both significant': '#FFD93D'
}

# Plot each category
for category in colors.keys():
    if category in plot_df['sig_category'].values:
        subset = plot_df[plot_df['sig_category'] == category]
        ax.scatter(subset['log2_gene_hr'], subset['log2_interaction_hr'],
                   c=colors[category], label=f'{category} (n={len(subset)})',
                   alpha=0.7, s=50, edgecolors='white', linewidth=0.5)

# Add reference lines
ax.axhline(y=0, color='black', linestyle='--', alpha=0.5, linewidth=1, label='No interaction effect')
ax.axvline(x=0, color='black', linestyle='--', alpha=0.5, linewidth=1, label='No gene effect')

# Label genes with significant interactions
sig_both = plot_df[plot_df['sig_category'] == 'Both significant']
sig_interaction_only = plot_df[plot_df['sig_category'] == 'Interaction only']

genes_to_label = pd.concat([sig_both, sig_interaction_only])

if len(genes_to_label) > 0:
    try:
        from adjustText import adjust_text
        texts = []
        for idx, row in genes_to_label.iterrows():
            texts.append(ax.text(
                row['log2_gene_hr'], row['log2_interaction_hr'], row['gene'],
                fontsize=9, fontweight='bold',
                bbox=dict(boxstyle="round,pad=0.3",
                          facecolor='white',
                          alpha=0.8,
                          edgecolor='black',
                          linewidth=0.5)))
        
        # Adjust text positions to avoid overlaps
        adjust_text(texts, ax=ax,
                    expand_points=(1.2, 1.2),
                    expand_text=(1.2, 1.2),
                    arrowprops=dict(arrowstyle='->', color='gray', alpha=0.7, lw=0.5))
    except ImportError:
        print("Note: adjustText not available, labels may overlap")
        for idx, row in genes_to_label.head(20).iterrows():
            ax.annotate(row['gene'],
                        (row['log2_gene_hr'], row['log2_interaction_hr']),
                        xytext=(5, 5), textcoords='offset points',
                        fontsize=8, fontweight='bold',
                        bbox=dict(boxstyle="round,pad=0.3",
                                  facecolor="white",
                                  alpha=0.8,
                                  edgecolor='black',
                                  linewidth=0.5))

# Labels and formatting
ax.set_xlabel('log2(Gene Expression HR)', fontsize=12)
ax.set_ylabel('log2(Interaction HR: Gene × ANCESTRY.AFR)', fontsize=12)
ax.set_title(f'Gene Expression vs Interaction Effects on Survival\n{COMPARTMENT_DESCRIPTION}',
             fontsize=14, pad=20)

# Add correlation
correlation = plot_df['log2_gene_hr'].corr(plot_df['log2_interaction_hr'])
ax.text(0.05, 0.95, f'Correlation: r = {correlation:.3f}',
        transform=ax.transAxes, fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

# Legend
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir_fig_tables + f"{FILE_NAME}_gene_vs_interaction_HR.png", dpi=300, bbox_inches='tight')
plt.savefig(output_dir_fig_tables + f"{FILE_NAME}_gene_vs_interaction_HR.pdf", bbox_inches='tight')
plt.show()

print(f"\n✓ Saved plot to {output_dir_fig_tables}{FILE_NAME}_gene_vs_interaction_HR.png/pdf")

### Volcano Plot: Interaction Term Significance

Visualizing the significance and effect size of interaction terms across all genes

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Prepare volcano plot data for interaction terms
volcano_df = interaction_success.copy()
volcano_df['log2_interaction_hr'] = np.log2(volcano_df['interaction_hr'])
volcano_df['neg_log10_interaction_p'] = -np.log10(volcano_df['interaction_p'])

# Define significance
p_threshold = 0.05
hr_threshold = 0.2  # log2 scale

volcano_df['significant_p'] = volcano_df['interaction_p'] < p_threshold
volcano_df['significant_hr'] = np.abs(volcano_df['log2_interaction_hr']) >= hr_threshold
volcano_df['significant'] = volcano_df['significant_p'] & volcano_df['significant_hr']

# Create colors
colors = []
for _, row in volcano_df.iterrows():
    if not row['significant_p']:
        colors.append('lightgrey')
    elif not row['significant_hr']:
        colors.append('grey')
    elif row['log2_interaction_hr'] < -hr_threshold:
        colors.append('blue')  # Protective interaction
    elif row['log2_interaction_hr'] > hr_threshold:
        colors.append('red')   # Risk interaction
    else:
        colors.append('grey')

# Create plot
fig, ax = plt.subplots(figsize=(14, 10))

scatter = ax.scatter(volcano_df['log2_interaction_hr'], 
                     volcano_df['neg_log10_interaction_p'],
                     c=colors, alpha=0.6, s=30)

# Reference lines
p_line_y = -np.log10(p_threshold)
ax.axhline(y=p_line_y, color='gray', linestyle='--', alpha=0.7, label=f'p = {p_threshold}')
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.7, label='HR = 1')
ax.axvline(x=hr_threshold, color='darkgray', linestyle=':', alpha=0.7, 
           label=f'HR = {2**hr_threshold:.2f}')
ax.axvline(x=-hr_threshold, color='darkgray', linestyle=':', alpha=0.7)

# Label top significant genes
significant_genes = volcano_df[volcano_df['significant']]
if len(significant_genes) > 0:
    top_genes = significant_genes.nsmallest(min(30, len(significant_genes)), 'interaction_p')
    
    try:
        from adjustText import adjust_text
        texts = []
        for idx, row in top_genes.iterrows():
            texts.append(ax.text(row['log2_interaction_hr'], row['neg_log10_interaction_p'], row['gene'],
                                 fontsize=8, alpha=0.8,
                                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7)))
        
        adjust_text(texts, arrowprops=dict(arrowstyle='-', color='black', alpha=0.5),
                    force_points=0.2, force_text=0.2)
    except ImportError:
        for idx, row in top_genes.head(15).iterrows():
            ax.annotate(row['gene'], (row['log2_interaction_hr'], row['neg_log10_interaction_p']),
                        xytext=(5, 5), textcoords='offset points',
                        fontsize=8, alpha=0.8,
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

# Formatting
ax.set_xlabel('log2(Interaction HR)', fontsize=12)
ax.set_ylabel('-log10(Interaction P-value)', fontsize=12)
ax.set_title(f'Volcano Plot: Gene × ANCESTRY.AFR Interaction Effects\n{COMPARTMENT_DESCRIPTION}', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='blue', label=f'Significant Protective Interaction (HR < {2**-hr_threshold:.2f})'),
    Patch(facecolor='red', label=f'Significant Risk Interaction (HR > {2**hr_threshold:.2f})'),
    Patch(facecolor='grey', label='Significant p, small effect'),
    Patch(facecolor='lightgrey', label='Not significant'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9)

# Summary stats
n_sig_p = (volcano_df['interaction_p'] < p_threshold).sum()
n_sig_full = volcano_df['significant'].sum()
n_protective = ((volcano_df['log2_interaction_hr'] < -hr_threshold) & 
                (volcano_df['interaction_p'] < p_threshold)).sum()
n_risk = ((volcano_df['log2_interaction_hr'] > hr_threshold) & 
          (volcano_df['interaction_p'] < p_threshold)).sum()

stats_text = f'Significant p-value (< {p_threshold}): {n_sig_p}\n'
stats_text += f'Significant p + effect (|log2HR| ≥ {hr_threshold}): {n_sig_full}\n'
stats_text += f'Protective interaction: {n_protective}, Risk interaction: {n_risk}'

ax.text(0.02, 0.98, stats_text, transform=ax.transAxes,
        verticalalignment='top', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.5', facecolor='lightgray', alpha=0.8))

plt.tight_layout()
plt.savefig(output_dir_fig_tables + f"{FILE_NAME}_interaction_volcano.png", dpi=300, bbox_inches='tight')
plt.savefig(output_dir_fig_tables + f"{FILE_NAME}_interaction_volcano.pdf", bbox_inches='tight')
plt.show()

print(f"\n✓ Saved plot to {output_dir_fig_tables}{FILE_NAME}_interaction_volcano.png/pdf")

### Comparison with Differential Expression Analysis

Compare interaction effects with differential expression between AFR high vs low groups

In [ ]:
# Merge interaction results with DEG results
if 'merged_df' in globals() and merged_df is not None:
    # Prepare DEG data
    deg_df = merged_df.copy()
    deg_df['gene'] = deg_df.index
    
    # Merge with interaction results
    comparison_df = pd.merge(
        interaction_success[['gene', 'gene_hr', 'gene_p', 'gene_fdr_p', 
                             'interaction_hr', 'interaction_p', 'interaction_fdr_p',
                             'covariate_hr', 'covariate_p']].set_index('gene'),
        deg_df[['log2FoldChange', 'padj']],
        left_index=True,
        right_index=True,
        how='inner'
    )
    
    print(f"Merged {len(comparison_df)} genes with both interaction and DEG data")
    
    # Identify genes significant in both analyses
    sig_both = comparison_df[
        (comparison_df['interaction_p'] < 0.05) & 
        (comparison_df['padj'] < 0.05)
    ]
    
    print(f"\nGenes significant for both interaction term AND differential expression: {len(sig_both)}")
    
    if len(sig_both) > 0:
        print("\nTop genes significant in both analyses:")
        print(sig_both.sort_values('interaction_p').head(15)[
            ['log2FoldChange', 'padj', 'interaction_hr', 'interaction_p', 'gene_hr', 'gene_p']
        ])
        
        # Create scatter plot
        fig, ax = plt.subplots(figsize=(12, 10))
        
        # Define categories
        comparison_df['category'] = 'Not significant'
        comparison_df.loc[
            (comparison_df['interaction_p'] < 0.05) & (comparison_df['padj'] >= 0.05),
            'category'
        ] = 'Interaction only'
        comparison_df.loc[
            (comparison_df['interaction_p'] >= 0.05) & (comparison_df['padj'] < 0.05),
            'category'
        ] = 'DEG only'
        comparison_df.loc[
            (comparison_df['interaction_p'] < 0.05) & (comparison_df['padj'] < 0.05),
            'category'
        ] = 'Both significant'
        
        # Colors
        cat_colors = {
            'Not significant': '#CCCCCC',
            'Interaction only': '#FF6B6B',
            'DEG only': '#4ECDC4',
            'Both significant': '#FFD93D'
        }
        
        # Plot
        for category in cat_colors.keys():
            if category in comparison_df['category'].values:
                subset = comparison_df[comparison_df['category'] == category]
                ax.scatter(subset['log2FoldChange'], 
                          np.log2(subset['interaction_hr']),
                          c=cat_colors[category], 
                          label=f'{category} (n={len(subset)})',
                          alpha=0.7, s=50, edgecolors='white', linewidth=0.5)
        
        # Reference lines
        ax.axhline(y=0, color='black', linestyle='--', alpha=0.5, linewidth=1)
        ax.axvline(x=0, color='black', linestyle='--', alpha=0.5, linewidth=1)
        
        # Label significant genes
        if len(sig_both) > 0:
            try:
                from adjustText import adjust_text
                texts = []
                for gene, row in sig_both.head(20).iterrows():
                    texts.append(ax.text(
                        row['log2FoldChange'], 
                        np.log2(row['interaction_hr']), 
                        gene,
                        fontsize=8, fontweight='bold',
                        bbox=dict(boxstyle="round,pad=0.3", facecolor='white', 
                                 alpha=0.8, edgecolor='black', linewidth=0.5)))
                
                adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='->', 
                           color='gray', alpha=0.7, lw=0.5))
            except ImportError:
                for gene, row in sig_both.head(10).iterrows():
                    ax.annotate(gene, (row['log2FoldChange'], np.log2(row['interaction_hr'])),
                               xytext=(5, 5), textcoords='offset points', fontsize=8)
        
        ax.set_xlabel('log2FoldChange (AFR High vs Low - DEG)', fontsize=12)
        ax.set_ylabel('log2(Interaction HR)', fontsize=12)
        ax.set_title(f'Differential Expression vs Interaction Effects\n{COMPARTMENT_DESCRIPTION}',
                    fontsize=14, pad=20)
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(output_dir_fig_tables + f"{FILE_NAME}_DEG_vs_interaction.png", 
                   dpi=300, bbox_inches='tight')
        plt.savefig(output_dir_fig_tables + f"{FILE_NAME}_DEG_vs_interaction.pdf", 
                   bbox_inches='tight')
        plt.show()
        
        print(f"\n✓ Saved plot to {output_dir_fig_tables}{FILE_NAME}_DEG_vs_interaction.png/pdf")
else:
    print("DEG results not available for comparison")

### Summary Table: Top Interaction Genes

Export a summary table of genes with the most significant interactions

In [ ]:
# Create summary table of top interaction genes
if len(sig_interaction) > 0:
    summary_table = sig_interaction.sort_values('interaction_p').copy()
    
    # Select key columns
    summary_cols = [
        'gene', 'n_patients', 'n_events',
        'gene_hr', 'gene_p', 'gene_fdr_p',
        'covariate_hr', 'covariate_p', 
        'interaction_hr', 'interaction_p', 'interaction_fdr_p'
    ]
    
    summary_export = summary_table.reset_index()[summary_cols].head(50)
    
    # Round numeric columns
    numeric_cols = summary_export.select_dtypes(include=[np.number]).columns
    summary_export[numeric_cols] = summary_export[numeric_cols].round(4)
    
    # Save to file
    summary_file = output_dir_fig_tables + f"{FILE_NAME}_top_interaction_genes.csv"
    summary_export.to_csv(summary_file, index=False)
    print(f"✓ Saved top 50 interaction genes to {summary_file}")
    
    # Display top 20
    print("\n=== TOP 20 GENES WITH SIGNIFICANT INTERACTIONS ===")
    display(summary_export.head(20))
else:
    print("No significant interactions to export")

### Interpretation Guide

**Understanding the Interaction Analysis:**

The interaction term (`gene × ANCESTRY.AFR`) tests whether the survival effect of gene expression differs based on African ancestry level.

**Interaction HR Interpretation:**
- **Interaction HR > 1**: The gene's effect on survival risk is amplified in patients with higher African ancestry
- **Interaction HR < 1**: The gene's effect on survival risk is reduced in patients with higher African ancestry

**Example Scenarios:**

1. **Gene HR = 2.0, Interaction HR = 1.5**
   - Gene doubles risk overall
   - This risk effect is 50% stronger in high AFR patients
   
2. **Gene HR = 0.5, Interaction HR = 0.7**
   - Gene is protective (halves risk) overall
   - This protective effect is 30% weaker in high AFR patients
   
3. **Gene HR = 2.0, Interaction HR = 0.5**
   - Gene doubles risk overall
   - But this risk is halved in high AFR patients (suggesting ancestry-specific protective mechanisms)

**Clinical Relevance:**
Genes with significant interactions may represent:
- Ancestry-specific therapeutic targets
- Biomarkers requiring ancestry-stratified interpretation
- Biological pathways with differential regulation across ancestries